<a href="https://colab.research.google.com/github/ArghyaRC96/metricguard-ai/blob/main/notebooks/08_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 MetricGuard AI — Formal Evaluation

## Notebook 08 — Retrieval, Reasoning & Safety Evaluation

This notebook evaluates the production MetricGuard Agentic RAG system
against the quarantined synthetic ground-truth dataset.

Ground-truth files are used only for evaluation.

They are never:

- embedded
- indexed
- inserted into Qdrant
- retrieved as runtime evidence
- shown to the LLM as knowledge-base context

### Evaluation Areas

1. retrieval relevance
2. diagnosis correctness
3. metric-conflict detection
4. stale-definition detection
5. intentional semantic-difference recognition
6. unsupported-query rejection
7. evidence/source correctness
8. confidence behaviour
9. revision behaviour
10. runtime latency and cost characteristics

In [1]:
from pathlib import Path
import shutil
import subprocess

GITHUB_USERNAME = "ArghyaRC96"

REPO_URL = (
    f"https://github.com/"
    f"{GITHUB_USERNAME}/metricguard-ai.git"
)

REPO_DIR = Path(
    "/content/metricguard-ai"
)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git",
        "clone",
        REPO_URL,
        str(REPO_DIR),
    ],
    check=True,
)

print("Repository:", REPO_DIR)

Repository: /content/metricguard-ai


## Fetching latest production code

In [2]:
import subprocess

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "pull",
        "origin",
        "main",
    ],
    check=True,
)

print("✅ Latest MetricGuard production code pulled.")

✅ Latest MetricGuard production code pulled.


In [3]:
%pip install -q -e "/content/metricguard-ai"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 19.1 MB/s eta 0:00:00
  Building editable for metricguard-ai (pyproject.toml) ... done


In [4]:
from qdrant_client import (
    QdrantClient,
    models,
)

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder,
)

import langgraph

print("✅ qdrant-client ready.")
print("✅ sentence-transformers ready.")
print("✅ LangGraph ready.")

✅ qdrant-client ready.
✅ sentence-transformers ready.
✅ LangGraph ready.


In [5]:
import sys

SOURCE_DIR = (
    REPO_DIR
    / "src"
)

if str(SOURCE_DIR) not in sys.path:
    sys.path.append(
        str(SOURCE_DIR)
    )

print(SOURCE_DIR)

/content/metricguard-ai/src


In [6]:
from metricguard.agents import (
    EvidenceRetrievalAgent,
    MetricInvestigator,
    MetricInvestigationAgent,
    VerificationReporter,
    VerificationReportingAgent,
    MetricGuardAgentSystem,
    build_metricguard_agent_graph,
    load_agent_config,
)

print("✅ Production agent modules imported.")

✅ Production agent modules imported.


In [7]:
agent_config = load_agent_config(
    REPO_DIR
)

print("✅ Agent config loaded.")
print("Max revisions:", agent_config.max_revisions)

✅ Agent config loaded.
Max revisions: 1


## Load API Key

In [9]:
from google.colab import userdata
from google import genai

# Load Gemini API key securely from Colab Secrets
GEMINI_API_KEY = userdata.get(
    "GEMINI_API_KEY"
)

assert GEMINI_API_KEY, (
    "GEMINI_API_KEY was not found "
    "in Colab Secrets."
)

# Model used by MetricGuard
GEMINI_MODEL = "gemini-3.7-flash"

# Create Gemini client
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("✅ Gemini client created.")
print("Model:", GEMINI_MODEL)

✅ Gemini client created.
Model: gemini-3.7-flash


In [10]:
import time

# ---------------------------------------------------------
# GEMINI RATE LIMIT SETTINGS
# ---------------------------------------------------------

# AI Studio currently shows 5 requests per minute.
GEMINI_RPM_LIMIT = 5

# 60 / 5 = 12 seconds.
# Add 2 seconds as a safety buffer.
MIN_GEMINI_INTERVAL = 14

_last_gemini_call = 0.0


# ---------------------------------------------------------
# DEVELOPMENT CACHE
# ---------------------------------------------------------

# Stores already-generated answers during this Colab session.
# If the exact same question is asked again,
# Gemini will NOT be called again.
_gemini_answer_cache = {}


# ---------------------------------------------------------
# RATE LIMIT HELPER
# ---------------------------------------------------------

def wait_for_gemini_slot():
    global _last_gemini_call

    elapsed = (
        time.monotonic()
        - _last_gemini_call
    )

    wait_time = max(
        0,
        MIN_GEMINI_INTERVAL - elapsed,
    )

    if wait_time > 0:
        print(
            f"⏳ Gemini rate-limit safety wait: "
            f"{wait_time:.1f} seconds"
        )

        time.sleep(
            wait_time
        )

    _last_gemini_call = (
        time.monotonic()
    )


print("✅ Gemini rate limiter ready.")
print("✅ Development cache ready.")

✅ Gemini rate limiter ready.
✅ Development cache ready.


In [11]:
wait_for_gemini_slot()

interaction = (
    gemini_client
    .interactions
    .create(
        model=GEMINI_MODEL,
        input=(
            "Reply with exactly: "
            "GEMINI_OK"
        ),
    )
)

print(
    interaction.output_text
)

GEMINI_OK


In [12]:
from metricguard.llm import (
    build_structured_llm,
)

agent_llm = build_structured_llm(
    repo_root=REPO_DIR,
    client=gemini_client,
    minimum_request_interval_seconds=14.0,
)

print("✅ Shared production Gemini adapter ready.")

✅ Shared production Gemini adapter ready.


## Regenerating Chunks

In [13]:
from datetime import date

from metricguard.lineage.enrichment_pipeline import (
    run_full_knowledge_enrichment,
)

run_full_knowledge_enrichment(
    REPO_DIR,
    as_of_date=date(
        2026,
        8,
        18,
    ),
)

METRICGUARD FULL KNOWLEDGE ENRICHMENT REPORT
Parsed documents      : 55
Final chunks          : 167
Metric-aware chunks   : 97
Lineage-aware chunks  : 90
Lineage graph nodes   : 33
Lineage graph edges   : 30
Freshness as-of       : 2026-08-18
Ground truth          : excluded
Embedding readiness   : YES


In [14]:
import json

CHUNKS_PATH = (
    REPO_DIR
    / "data"
    / "processed"
    / "fully_enriched_chunks.jsonl"
)

chunks = []

with CHUNKS_PATH.open(
    "r",
    encoding="utf-8",
) as file:

    for line in file:
        chunks.append(
            json.loads(line)
        )

print(
    "Chunks:",
    len(chunks)
)

Chunks: 167


In [15]:
assert not any(
    "ground_truth"
    in chunk[
        "metadata"
    ].get(
        "source_path",
        "",
    )
    for chunk in chunks
)

print(
    "✅ Ground truth excluded."
)

✅ Ground truth excluded.


In [16]:
from metricguard.retrieval import (
    CrossEncoderReranker,
    DenseRetriever,
    RetrievalPipeline,
    format_final_evidence,
    load_embedding_model,
    load_reranker_model,
    load_retrieval_config,
)

retrieval_config = (
    load_retrieval_config(
        REPO_DIR
    )
)

retrieval_config

RetrievalConfig(candidate_top_k=20, final_top_k=5, reranker_model='cross-encoder/ms-marco-MiniLM-L6-v2', embedding_model='sentence-transformers/all-mpnet-base-v2', collection_name='metricguard_dense_v1', normalize_embeddings=True)

In [17]:
embedding_model = (
    load_embedding_model(
        retrieval_config
        .embedding_model
    )
)

VECTOR_SIZE = (
    embedding_model
    .get_embedding_dimension()
)

print(
    "Vector size:",
    VECTOR_SIZE
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector size: 768


In [18]:
def build_embedding_text(
    chunk: dict,
) -> str:

    metadata = chunk[
        "metadata"
    ]

    parts = [
        (
            "Source type: "
            f"{metadata.get('source_type')}"
        ),
        (
            "Asset type: "
            f"{metadata.get('asset_type')}"
        ),
        (
            "File: "
            f"{metadata.get('file_name')}"
        ),
    ]

    for label, key in [
        ("Metric", "metric_name"),
        (
            "Observed version",
            "observed_version",
        ),
        (
            "Authoritative version",
            "authoritative_version",
        ),
        (
            "Version relation",
            "version_relation",
        ),
        (
            "Freshness",
            "freshness_status",
        ),
    ]:

        value = metadata.get(key)

        if value:
            parts.append(
                f"{label}: {value}"
            )

    return (
        "\n".join(parts)
        + "\n\n"
        + chunk["content"]
    )

In [19]:
embedding_texts = [
    build_embedding_text(
        chunk
    )
    for chunk in chunks
]

embeddings = (
    embedding_model.encode(
        embedding_texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
)

print(
    embeddings.shape
)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

(167, 768)


In [20]:
from qdrant_client import (
    QdrantClient,
    models,
)

qdrant_client = (
    QdrantClient(
        ":memory:"
    )
)

COLLECTION_NAME = (
    retrieval_config
    .collection_name
)

qdrant_client.create_collection(
    collection_name=
        COLLECTION_NAME,
    vectors_config=
        models.VectorParams(
            size=VECTOR_SIZE,
            distance=
                models.Distance.COSINE,
        ),
)

print(
    "✅ Qdrant ready."
)

✅ Qdrant ready.


In [21]:
import uuid

points = []

for chunk, vector in zip(
    chunks,
    embeddings,
):

    point_id = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            chunk["chunk_id"],
        )
    )

    payload = {
        "chunk_id":
            chunk["chunk_id"],
        "content":
            chunk["content"],
        **chunk["metadata"],
    }

    points.append(
        models.PointStruct(
            id=point_id,
            vector=
                vector.tolist(),
            payload=payload,
        )
    )

In [22]:
qdrant_client.upsert(
    collection_name=
        COLLECTION_NAME,
    points=points,
    wait=True,
)

print(
    "Qdrant points:",
    qdrant_client
    .get_collection(
        COLLECTION_NAME
    )
    .points_count,
)

Qdrant points: 167


## Building Production Retriever

In [23]:
production_dense = (
    DenseRetriever(
        client=qdrant_client,
        embedding_model=
            embedding_model,
        collection_name=
            COLLECTION_NAME,
        normalize_embeddings=True,
    )
)

In [24]:
reranker_model = (
    load_reranker_model(
        retrieval_config
        .reranker_model
    )
)

production_reranker = (
    CrossEncoderReranker(
        model=reranker_model,
        batch_size=16,
    )
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [25]:
production_retrieval = (
    RetrievalPipeline(
        dense_retriever=
            production_dense,
        reranker=
            production_reranker,
        candidate_top_k=
            retrieval_config
            .candidate_top_k,
        final_top_k=
            retrieval_config
            .final_top_k,
    )
)

print(
    "✅ Production retrieval ready."
)

✅ Production retrieval ready.


## Agentic RAG Integration

In [26]:
required_objects = {
    "REPO_DIR": "REPO_DIR" in globals(),
    "gemini_client": "gemini_client" in globals(),
    "production_retrieval": "production_retrieval" in globals(),
}

for name, exists in required_objects.items():
    print(
        "✅" if exists else "❌",
        name
    )

missing = [
    name
    for name, exists
    in required_objects.items()
    if not exists
]

assert not missing, (
    "Missing prerequisite objects: "
    + ", ".join(missing)
)

print(
    "\n🔥 Phase 8.3 prerequisites are ready."
)

✅ REPO_DIR
✅ gemini_client
✅ production_retrieval

🔥 Phase 8.3 prerequisites are ready.


In [27]:
import langgraph

print(
    "✅ Editable MetricGuard package refreshed."
)
print(
    "✅ LangGraph import successful."
)

✅ Editable MetricGuard package refreshed.
✅ LangGraph import successful.


In [28]:
agent_config = load_agent_config(
    REPO_DIR
)

print(
    "✅ Agent configuration loaded."
)

print(
    "Max revisions:",
    agent_config.max_revisions
)

✅ Agent configuration loaded.
Max revisions: 1


In [29]:
agent_llm = build_structured_llm(
    repo_root=REPO_DIR,
    client=gemini_client,
    minimum_request_interval_seconds=14.0,
)

print(
    "✅ Shared Gemini agent LLM ready."
)

✅ Shared Gemini agent LLM ready.


### Building Agent 1

In [30]:
evidence_agent = EvidenceRetrievalAgent(
    retrieval_pipeline=
        production_retrieval
)

print(
    "✅ Agent 1: Evidence Retrieval ready."
)

✅ Agent 1: Evidence Retrieval ready.


### Building Agent 2

In [31]:
investigator = MetricInvestigator(
    llm=agent_llm
)

investigation_agent = (
    MetricInvestigationAgent(
        investigator=investigator
    )
)

print(
    "✅ Agent 2: Metric Investigation ready."
)

✅ Agent 2: Metric Investigation ready.


### Building Agent 3

In [32]:
verifier = VerificationReporter(
    llm=agent_llm
)

verification_agent = (
    VerificationReportingAgent(
        verifier=verifier
    )
)

print(
    "✅ Agent 3: Verification & Reporting ready."
)

✅ Agent 3: Verification & Reporting ready.


### Compiling Langgraph

In [33]:
agent_graph = (
    build_metricguard_agent_graph(
        evidence_agent=
            evidence_agent,

        investigation_agent=
            investigation_agent,

        verification_agent=
            verification_agent,
    )
)

print(
    "✅ MetricGuard LangGraph compiled."
)

✅ MetricGuard LangGraph compiled.


In [34]:
agent_system = (
    MetricGuardAgentSystem(
        graph=agent_graph,
        max_revisions=
            agent_config.max_revisions,
    )
)

print(
    "🔥 MetricGuard Agent System ready."
)

🔥 MetricGuard Agent System ready.


### Validation

In [35]:
checks = {
    "REPO_DIR":
        REPO_DIR is not None,

    "Gemini client":
        gemini_client is not None,

    "Production retrieval":
        production_retrieval is not None,

    "Shared agent LLM":
        agent_llm is not None,

    "Agent 1":
        evidence_agent is not None,

    "Agent 2":
        investigation_agent is not None,

    "Agent 3":
        verification_agent is not None,

    "LangGraph":
        agent_graph is not None,

    "Agent system":
        agent_system is not None,
}


for name, ok in checks.items():

    print(
        "✅" if ok else "❌",
        name
    )


assert all(
    checks.values()
)

print(
    "\n🔥 FULL PHASE 8.3 ASSEMBLY PASSED."
)

✅ REPO_DIR
✅ Gemini client
✅ Production retrieval
✅ Shared agent LLM
✅ Agent 1
✅ Agent 2
✅ Agent 3
✅ LangGraph
✅ Agent system

🔥 FULL PHASE 8.3 ASSEMBLY PASSED.


In [36]:
agentic_result_cache = {}


def run_agent_question(question: str):
    normalized_question = " ".join(
        question.strip().split()
    )

    if normalized_question in agentic_result_cache:
        print(
            "✅ Using cached agentic result. "
            "No new Gemini calls."
        )

        return agentic_result_cache[
            normalized_question
        ]

    print(
        "🔥 Running real MetricGuard "
        "agentic investigation..."
    )

    result = agent_system.investigate(
        normalized_question
    )

    agentic_result_cache[
        normalized_question
    ] = result

    return result


print(
    "✅ Notebook agentic cache ready."
)

✅ Notebook agentic cache ready.


In [37]:
import json


def show_agent_result(result):
    print("=" * 80)
    print("FINAL DECISION")
    print("=" * 80)

    print(
        result.get(
            "verification_decision"
        )
    )

    print("\n" + "=" * 80)
    print("REVISION COUNT")
    print("=" * 80)

    print(
        result.get(
            "revision_count",
            0
        )
    )

    print("\n" + "=" * 80)
    print("LANGGRAPH TRACE")
    print("=" * 80)

    for step in result.get(
        "trace",
        []
    ):
        print("->", step)

    print("\n" + "=" * 80)
    print("AGENT 2 INVESTIGATION")
    print("=" * 80)

    print(
        json.dumps(
            result.get(
                "investigation",
                {}
            ),
            indent=2,
            default=str
        )
    )

    print("\n" + "=" * 80)
    print("AGENT 3 FINAL REPORT")
    print("=" * 80)

    print(
        json.dumps(
            result.get(
                "final_report",
                {}
            ),
            indent=2,
            default=str
        )
    )

In [38]:
def validate_agent_result(
    result,
    label: str
):
    decision = result.get(
        "verification_decision"
    )

    report = result.get(
        "final_report",
        {}
    )

    evidence = result.get(
        "evidence",
        []
    )

    revision_count = int(
        result.get(
            "revision_count",
            0
        )
    )

    assert result.get(
        "retrieval_complete"
    ) is True

    assert decision in {
        "approved",
        "insufficient_evidence",
    }

    assert (
        revision_count
        <= agent_config.max_revisions
    )

    assert (
        report.get("decision")
        == decision
    )

    confidence = float(
        report.get(
            "confidence",
            0.0
        )
    )

    assert 0.0 <= confidence <= 1.0

    allowed_ids = {
        f"E{i}"
        for i in range(
            1,
            len(evidence) + 1
        )
    }

    used_ids = set(
        report.get(
            "evidence_ids",
            []
        )
    )

    assert used_ids.issubset(
        allowed_ids
    )

    if decision == "approved":
        assert used_ids

    for item in evidence:
        source_path = str(
            item.get(
                "source_path",
                ""
            )
        )

        assert (
            "ground_truth"
            not in source_path
        )

    print(
        f"✅ {label}: structural validation passed."
    )

---

## Phase 9.1 — Load Quarantined Ground Truth

Ground truth is loaded only after the production MetricGuard runtime has
already been constructed.

This prevents evaluation answers from contaminating retrieval or generation.

In [39]:
from pathlib import Path


GROUND_TRUTH_DIR = (
    REPO_DIR
    / "data"
    / "ground_truth"
)


assert (
    GROUND_TRUTH_DIR.exists()
), (
    "Ground-truth directory not found."
)


print(
    "✅ Ground-truth directory located."
)

print(
    "Path:",
    GROUND_TRUTH_DIR
)

✅ Ground-truth directory located.
Path: /content/metricguard-ai/data/ground_truth


In [40]:
ground_truth_files = sorted(
    path.name
    for path
    in GROUND_TRUTH_DIR.iterdir()
    if path.is_file()
)


for file_name in ground_truth_files:
    print(
        "✅",
        file_name
    )

✅ evaluation_questions.json
✅ expected_answers.json
✅ expected_lineage.json
✅ expected_versions.json
✅ known_conflicts.json


In [42]:
import json


evaluation_questions_path = (
    GROUND_TRUTH_DIR
    / "evaluation_questions.json"
)


with evaluation_questions_path.open(
    "r",
    encoding="utf-8-sig",
) as file:

    evaluation_questions = (
        json.load(file)
    )


print(
    "✅ Evaluation questions loaded."
)

print(
    "Type:",
    type(evaluation_questions).__name__
)

print(
    "Count:",
    len(evaluation_questions)
)

✅ Evaluation questions loaded.
Type: list
Count: 10


In [43]:
expected_answers_path = (
    GROUND_TRUTH_DIR
    / "expected_answers.json"
)


with expected_answers_path.open(
    "r",
    encoding="utf-8-sig",
) as file:

    expected_answers = (
        json.load(file)
    )


print(
    "✅ Expected answers loaded."
)

print(
    "Type:",
    type(expected_answers).__name__
)

print(
    "Count:",
    len(expected_answers)
)

✅ Expected answers loaded.
Type: list
Count: 10


In [44]:
print(
    json.dumps(
        evaluation_questions,
        indent=2
    )[:4000]
)

[
  {
    "question_id": "Q001",
    "category": "conflict_detection",
    "question": "Why does the Executive KPI Dashboard report different Net Revenue from the Finance Revenue Dashboard after April 1, 2026?"
  },
  {
    "question_id": "Q002",
    "category": "version_detection",
    "question": "What is the current approved version of Net Revenue and when did it become effective?"
  },
  {
    "question_id": "Q003",
    "category": "staleness_detection",
    "question": "Is the Executive KPI Dashboard using the current Net Revenue definition?"
  },
  {
    "question_id": "Q004",
    "category": "semantic_conflict",
    "question": "Why does the Growth & Marketing Dashboard report more Active Customers than the enterprise KPI?"
  },
  {
    "question_id": "Q005",
    "category": "semantic_classification",
    "question": "Is the Total Orders disagreement between Operations and Finance a data pipeline failure?"
  },
  {
    "question_id": "Q006",
    "category": "version_detection",


In [45]:
print(
    json.dumps(
        expected_answers,
        indent=2
    )[:4000]
)

[
  {
    "question_id": "Q001",
    "required_facts": [
      "Finance uses Net Revenue v3.",
      "Executive uses Net Revenue v2.",
      "Net Revenue v3 became effective on 2026-04-01.",
      "Version 3 deducts posted chargebacks while Version 2 does not."
    ],
    "classification": "stale_version_conflict"
  },
  {
    "question_id": "Q002",
    "required_facts": [
      "The current Net Revenue version is v3.",
      "It became effective on 2026-04-01.",
      "The definition subtracts discounts, completed refunds, and posted chargebacks from Gross Revenue."
    ]
  },
  {
    "question_id": "Q003",
    "required_facts": [
      "No.",
      "The Executive KPI Dashboard uses Net Revenue v2.",
      "The current enterprise version is v3."
    ],
    "classification": "stale_version_conflict"
  },
  {
    "question_id": "Q004",
    "required_facts": [
      "Growth uses Active Customers v1.",
      "Version 1 counts identified customers with recent digital activity.",
      "The

---

## Phase 9.2 — Production Evaluation Run

The quarantined evaluation questions are now executed against the production
MetricGuard Agentic RAG service.

Ground truth is used only after each prediction is produced.

For every evaluation question, this phase records:

- question ID
- evaluation category
- production answer
- predicted diagnosis
- decision
- confidence
- resolved sources
- retrieval relevance score
- revision count
- graph trace
- runtime latency
- estimated LLM-call count

Scoring against expected facts is performed separately in Phase 9.3.

In [46]:
assert (
    "agent_system"
    in globals()
), (
    "agent_system is missing. "
    "Run the production agent assembly first."
)


if "metricguard" not in globals():

    from metricguard.agents.cache import (
        AgenticResultCache,
    )

    from metricguard.agents.service import (
        MetricGuardAgenticRAG,
        load_agentic_service_config,
    )

    agentic_service_config = (
        load_agentic_service_config(
            REPO_DIR
        )
    )

    production_agent_cache = (
        AgenticResultCache(
            max_entries=
                agentic_service_config
                .cache_max_entries
        )
        if agentic_service_config
           .cache_enabled
        else None
    )

    metricguard = (
        MetricGuardAgenticRAG(
            agent_system=
                agent_system,

            config=
                agentic_service_config,

            cache=
                production_agent_cache,
        )
    )


print(
    "🔥 Production metricguard service ready."
)

🔥 Production metricguard service ready.


In [47]:
question_lookup = {
    item["question_id"]:
        item
    for item
    in evaluation_questions
}


answer_lookup = {
    item["question_id"]:
        item
    for item
    in expected_answers
}


assert (
    set(question_lookup)
    == set(answer_lookup)
), (
    "Question IDs and expected-answer IDs "
    "do not match."
)


evaluation_cases = []


for question_id in sorted(
    question_lookup
):

    question_item = (
        question_lookup[
            question_id
        ]
    )

    expected_item = (
        answer_lookup[
            question_id
        ]
    )

    evaluation_cases.append(
        {
            "question_id":
                question_id,

            "category":
                question_item[
                    "category"
                ],

            "question":
                question_item[
                    "question"
                ],

            "required_facts":
                expected_item[
                    "required_facts"
                ],

            "expected_classification":
                expected_item.get(
                    "classification"
                ),
        }
    )


print(
    "✅ Evaluation cases assembled:",
    len(evaluation_cases)
)

✅ Evaluation cases assembled: 10


In [48]:
assert len(
    evaluation_cases
) == 10


for case in evaluation_cases:

    assert case[
        "question_id"
    ]

    assert case[
        "category"
    ]

    assert case[
        "question"
    ]

    assert (
        isinstance(
            case[
                "required_facts"
            ],
            list,
        )
    )

    assert (
        len(
            case[
                "required_facts"
            ]
        )
        > 0
    )


print(
    "✅ Ground-truth integrity validation passed."
)

✅ Ground-truth integrity validation passed.


In [49]:
import pandas as pd


evaluation_plan_df = (
    pd.DataFrame(
        [
            {
                "question_id":
                    case[
                        "question_id"
                    ],

                "category":
                    case[
                        "category"
                    ],

                "question":
                    case[
                        "question"
                    ],

                "required_fact_count":
                    len(
                        case[
                            "required_facts"
                        ]
                    ),

                "expected_classification":
                    case[
                        "expected_classification"
                    ],
            }
            for case
            in evaluation_cases
        ]
    )
)


evaluation_plan_df

,question_id,category,question,required_fact_count,expected_classification
0,Q001,conflict_detection,Why does the Executive KPI Dashboard report di...,4,stale_version_conflict
1,Q002,version_detection,What is the current approved version of Net Re...,3,None
2,Q003,staleness_detection,Is the Executive KPI Dashboard using the curre...,3,stale_version_conflict
3,Q004,semantic_conflict,Why does the Growth & Marketing Dashboard repo...,3,stale_semantic_definition
4,Q005,semantic_classification,Is the Total Orders disagreement between Opera...,4,expected_semantic_difference
5,Q006,version_detection,What is the current enterprise definition of T...,3,None
6,Q007,metric_migration,Why did Conversion Rate change in June 2026?,4,expected_metric_migration
7,Q008,lineage,What is the upstream lineage of Net Revenue sh...,4,None
8,Q009,impact_analysis,Which reporting asset is affected by the stale...,3,None
9,Q010,version_detection,What is the current Refund Rate definition?,4,None


In [50]:
EXPECTED_CLASSIFICATION_TO_DIAGNOSES = {

    "stale_version_conflict": {
        "version_mismatch",
        "stale_definition",
        "metric_migration",
    },

    "stale_semantic_definition": {
        "stale_definition",
        "version_mismatch",
    },

    "expected_semantic_difference": {
        "intentional_semantic_difference",
    },

    "expected_metric_migration": {
        "metric_migration",
    },
}


print(
    "✅ Evaluation classification mapping ready."
)

✅ Evaluation classification mapping ready.


In [51]:
if (
    getattr(
        metricguard,
        "cache",
        None,
    )
    is not None
):

    metricguard.cache.clear()


print(
    "✅ Evaluation starts with a clean production cache."
)

✅ Evaluation starts with a clean production cache.


In [52]:
import time


def estimate_llm_calls(
    trace: list[str],
) -> int:

    investigation_calls = (
        trace.count(
            "metric_investigation_agent"
        )
    )

    verification_calls = (
        trace.count(
            "verification_reporting_agent"
        )
    )

    return (
        investigation_calls
        + verification_calls
    )


def run_evaluation_case(
    case: dict,
) -> dict:

    question = (
        case[
            "question"
        ]
    )


    start_time = (
        time.perf_counter()
    )


    result = metricguard.ask(
        question
    )


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    sources = [
        source.model_dump()
        for source
        in result.sources
    ]


    return {
        "question_id":
            case[
                "question_id"
            ],

        "category":
            case[
                "category"
            ],

        "question":
            question,

        "expected_classification":
            case[
                "expected_classification"
            ],

        "required_facts":
            case[
                "required_facts"
            ],

        "status":
            result.status,

        "decision":
            result.decision,

        "predicted_diagnosis":
            result.diagnosis,

        "answer":
            result.answer,

        "key_findings":
            result.key_findings,

        "confidence":
            result.confidence,

        "sources":
            sources,

        "source_count":
            len(
                sources
            ),

        "retrieval_relevant":
            result
            .retrieval_relevant,

        "retrieval_top1_score":
            result
            .retrieval_top1_score,

        "revision_count":
            result.revision_count,

        "trace":
            result.trace,

        "estimated_llm_calls":
            estimate_llm_calls(
                result.trace
            ),

        "latency_seconds":
            elapsed_seconds,

        "cached":
            result.cached,
    }


print(
    "✅ Evaluation runner ready."
)

✅ Evaluation runner ready.


## Final Evaluation API Call

In [53]:
evaluation_results = []


for index, case in enumerate(
    evaluation_cases,
    start=1,
):

    print(
        "=" * 80
    )

    print(
        f"[{index}/{len(evaluation_cases)}]",
        case[
            "question_id"
        ],
        "-",
        case[
            "category"
        ],
    )

    print(
        case[
            "question"
        ]
    )


    try:

        result = (
            run_evaluation_case(
                case
            )
        )

        evaluation_results.append(
            result
        )

        print(
            "Decision:",
            result[
                "decision"
            ]
        )

        print(
            "Diagnosis:",
            result[
                "predicted_diagnosis"
            ]
        )

        print(
            "Confidence:",
            result[
                "confidence"
            ]
        )

        print(
            "Top-1 relevance:",
            result[
                "retrieval_top1_score"
            ]
        )

        print(
            "Revisions:",
            result[
                "revision_count"
            ]
        )

        print(
            "Estimated LLM calls:",
            result[
                "estimated_llm_calls"
            ]
        )

        print(
            "Latency:",
            round(
                result[
                    "latency_seconds"
                ],
                2,
            ),
            "seconds",
        )


    except Exception as exc:

        evaluation_results.append(
            {
                "question_id":
                    case[
                        "question_id"
                    ],

                "category":
                    case[
                        "category"
                    ],

                "question":
                    case[
                        "question"
                    ],

                "error":
                    repr(
                        exc
                    ),
            }
        )

        print(
            "❌ ERROR:",
            repr(
                exc
            )
        )

[1/10] Q001 - conflict_detection
Why does the Executive KPI Dashboard report different Net Revenue from the Finance Revenue Dashboard after April 1, 2026?
Decision: approved
Diagnosis: metric_migration
Confidence: 0.95
Top-1 relevance: None
Revisions: 0
Estimated LLM calls: 2
Latency: 48.48 seconds
[2/10] Q002 - version_detection
What is the current approved version of Net Revenue and when did it become effective?
Decision: approved
Diagnosis: metric_migration
Confidence: 1.0
Top-1 relevance: None
Revisions: 0
Estimated LLM calls: 2
Latency: 43.83 seconds
[3/10] Q003 - staleness_detection
Is the Executive KPI Dashboard using the current Net Revenue definition?
Decision: approved
Diagnosis: version_mismatch
Confidence: 0.95
Top-1 relevance: None
Revisions: 0
Estimated LLM calls: 2
Latency: 50.77 seconds
[4/10] Q004 - semantic_conflict
Why does the Growth & Marketing Dashboard report more Active Customers than the enterprise KPI?
Decision: approved
Diagnosis: intentional_semantic_differe

In [54]:
assert (
    len(
        evaluation_results
    )
    == len(
        evaluation_cases
    )
)


failed_cases = [
    item
    for item
    in evaluation_results
    if "error" in item
]


print(
    "Total cases:",
    len(
        evaluation_results
    )
)

print(
    "Execution errors:",
    len(
        failed_cases
    )
)


if failed_cases:

    for item in failed_cases:

        print(
            item[
                "question_id"
            ],
            item[
                "error"
            ]
        )

else:

    print(
        "🔥 All evaluation questions executed successfully."
    )

Total cases: 10
Execution errors: 0
🔥 All evaluation questions executed successfully.


In [55]:
evaluation_run_df = (
    pd.DataFrame(
        [
            {
                "question_id":
                    item.get(
                        "question_id"
                    ),

                "category":
                    item.get(
                        "category"
                    ),

                "decision":
                    item.get(
                        "decision"
                    ),

                "diagnosis":
                    item.get(
                        "predicted_diagnosis"
                    ),

                "confidence":
                    item.get(
                        "confidence"
                    ),

                "top1_relevance":
                    item.get(
                        "retrieval_top1_score"
                    ),

                "revisions":
                    item.get(
                        "revision_count"
                    ),

                "llm_calls":
                    item.get(
                        "estimated_llm_calls"
                    ),

                "latency_seconds":
                    item.get(
                        "latency_seconds"
                    ),

                "source_count":
                    item.get(
                        "source_count"
                    ),

                "error":
                    item.get(
                        "error"
                    ),
            }
            for item
            in evaluation_results
        ]
    )
)


evaluation_run_df

,question_id,category,decision,diagnosis,confidence,top1_relevance,revisions,llm_calls,latency_seconds,source_count,error
0,Q001,conflict_detection,approved,metric_migration,0.95,None,0,2,48.476258,5,None
1,Q002,version_detection,approved,metric_migration,1.00,None,0,2,43.830111,4,None
2,Q003,staleness_detection,approved,version_mismatch,0.95,None,0,2,50.765382,5,None
3,Q004,semantic_conflict,approved,intentional_semantic_difference,0.95,None,0,2,85.390344,5,None
4,Q005,semantic_classification,approved,intentional_semantic_difference,1.00,None,0,2,51.428417,5,None
5,Q006,version_detection,approved,metric_migration,0.95,None,0,2,148.333720,5,None
6,Q007,metric_migration,approved,metric_migration,0.98,None,0,2,82.660518,5,None
7,Q008,lineage,approved,metric_migration,0.95,None,0,2,46.883494,3,None
8,Q009,impact_analysis,approved,version_mismatch,0.95,None,0,2,53.615511,5,None
9,Q010,version_detection,approved,metric_migration,1.00,None,0,2,39.888605,5,None


---

## Phase 9.2A — Production Runtime Equivalence Check

The initial supported-query evaluation completed successfully, but the
retrieval relevance score was missing because Notebook 08 had retained the
pre-Phase-8.4 Agent 1 assembly.

The final production Agent 1 is rebuilt here with the calibrated relevance
gate before formal scoring.

Existing LLM answers are retained. Retrieval relevance is measured
deterministically without repeating the 10 Gemini evaluations.

In [56]:
from metricguard.retrieval.relevance import (
    RelevanceGate,
    load_relevance_config,
)

from metricguard.agents.retrieval_agent import (
    EvidenceRetrievalAgent,
)


relevance_config = (
    load_relevance_config(
        REPO_DIR
    )
)


relevance_gate = (
    RelevanceGate(
        enabled=
            relevance_config.enabled,

        threshold=
            relevance_config
            .top1_rerank_threshold,
    )
)


evidence_agent = (
    EvidenceRetrievalAgent(
        retrieval_pipeline=
            production_retrieval,

        relevance_gate=
            relevance_gate,
    )
)


print(
    "✅ Final production Agent 1 rebuilt."
)

print(
    "Relevance threshold:",
    relevance_config
    .top1_rerank_threshold
)

✅ Final production Agent 1 rebuilt.
Relevance threshold: 0.27


In [57]:
from metricguard.agents.graph import (
    build_metricguard_agent_graph,
)

from metricguard.agents.system import (
    MetricGuardAgentSystem,
)


agent_graph = (
    build_metricguard_agent_graph(
        evidence_agent=
            evidence_agent,

        investigation_agent=
            investigation_agent,

        verification_agent=
            verification_agent,
    )
)


agent_system = (
    MetricGuardAgentSystem(
        graph=
            agent_graph,

        max_revisions=
            agent_config.max_revisions,
    )
)


print(
    "✅ Final production LangGraph rebuilt."
)

✅ Final production LangGraph rebuilt.


In [58]:
from metricguard.agents.cache import (
    AgenticResultCache,
)

from metricguard.agents.service import (
    MetricGuardAgenticRAG,
    load_agentic_service_config,
)


agentic_service_config = (
    load_agentic_service_config(
        REPO_DIR
    )
)


production_agent_cache = (
    AgenticResultCache(
        max_entries=
            agentic_service_config
            .cache_max_entries
    )
)


metricguard = (
    MetricGuardAgenticRAG(
        agent_system=
            agent_system,

        config=
            agentic_service_config,

        cache=
            production_agent_cache,
    )
)


print(
    "🔥 Final production MetricGuard "
    "service rebuilt."
)

🔥 Final production MetricGuard service rebuilt.


In [59]:
retrieval_audit = {}


for case in evaluation_cases:

    question_id = (
        case[
            "question_id"
        ]
    )

    question = (
        case[
            "question"
        ]
    )


    retrieval_results = (
        production_retrieval
        .retrieve(
            question
        )
    )


    scores = [
        float(
            item.rerank_score
        )
        for item
        in retrieval_results
    ]


    top1_score = (
        max(scores)
        if scores
        else None
    )


    retrieval_audit[
        question_id
    ] = {
        "top1_score":
            top1_score,

        "retrieval_relevant":
            (
                top1_score
                is not None
                and
                top1_score
                >= relevance_config
                   .top1_rerank_threshold
            ),

        "top5_scores":
            scores[:5],
    }


print(
    "✅ Retrieval relevance audit complete."
)

✅ Retrieval relevance audit complete.


In [60]:
for result in evaluation_results:

    question_id = (
        result[
            "question_id"
        ]
    )

    audit = (
        retrieval_audit[
            question_id
        ]
    )

    result[
        "retrieval_top1_score"
    ] = audit[
        "top1_score"
    ]

    result[
        "retrieval_relevant"
    ] = audit[
        "retrieval_relevant"
    ]


print(
    "✅ Relevance scores attached "
    "to existing evaluation results."
)

✅ Relevance scores attached to existing evaluation results.


In [61]:
supported_relevance_scores = [
    result[
        "retrieval_top1_score"
    ]
    for result
    in evaluation_results
]


assert all(
    result[
        "retrieval_relevant"
    ]
    is True
    for result
    in evaluation_results
)


minimum_eval_relevance = min(
    supported_relevance_scores
)


print(
    "Minimum evaluation Top-1 score:",
    minimum_eval_relevance
)

print(
    "Production threshold:",
    relevance_config
    .top1_rerank_threshold
)

print(
    "Margin above threshold:",
    (
        minimum_eval_relevance
        -
        relevance_config
        .top1_rerank_threshold
    )
)

print(
    "\n🔥 ALL 10 SUPPORTED QUESTIONS "
    "PASSED THE PRODUCTION RELEVANCE GATE."
)

Minimum evaluation Top-1 score: 0.8079553842544556
Production threshold: 0.27
Margin above threshold: 0.5379553842544555

🔥 ALL 10 SUPPORTED QUESTIONS PASSED THE PRODUCTION RELEVANCE GATE.


In [62]:
evaluation_run_df = (
    pd.DataFrame(
        [
            {
                "question_id":
                    item.get(
                        "question_id"
                    ),

                "category":
                    item.get(
                        "category"
                    ),

                "decision":
                    item.get(
                        "decision"
                    ),

                "diagnosis":
                    item.get(
                        "predicted_diagnosis"
                    ),

                "confidence":
                    item.get(
                        "confidence"
                    ),

                "top1_relevance":
                    item.get(
                        "retrieval_top1_score"
                    ),

                "revisions":
                    item.get(
                        "revision_count"
                    ),

                "llm_calls":
                    item.get(
                        "estimated_llm_calls"
                    ),

                "latency_seconds":
                    item.get(
                        "latency_seconds"
                    ),

                "source_count":
                    item.get(
                        "source_count"
                    ),

                "error":
                    item.get(
                        "error"
                    ),
            }
            for item
            in evaluation_results
        ]
    )
)


evaluation_run_df

,question_id,category,decision,diagnosis,confidence,top1_relevance,revisions,llm_calls,latency_seconds,source_count,error
0,Q001,conflict_detection,approved,metric_migration,0.95,0.993834,0,2,48.476258,5,None
1,Q002,version_detection,approved,metric_migration,1.00,0.987470,0,2,43.830111,4,None
2,Q003,staleness_detection,approved,version_mismatch,0.95,0.998704,0,2,50.765382,5,None
3,Q004,semantic_conflict,approved,intentional_semantic_difference,0.95,0.992669,0,2,85.390344,5,None
4,Q005,semantic_classification,approved,intentional_semantic_difference,1.00,0.932100,0,2,51.428417,5,None
5,Q006,version_detection,approved,metric_migration,0.95,0.996773,0,2,148.333720,5,None
6,Q007,metric_migration,approved,metric_migration,0.98,0.995834,0,2,82.660518,5,None
7,Q008,lineage,approved,metric_migration,0.95,0.807955,0,2,46.883494,3,None
8,Q009,impact_analysis,approved,version_mismatch,0.95,0.920117,0,2,53.615511,5,None
9,Q010,version_detection,approved,metric_migration,1.00,0.989857,0,2,39.888605,5,None


In [63]:
production_smoke_result = (
    metricguard.ask(
        evaluation_cases[0][
            "question"
        ]
    )
)


print(
    "Decision:",
    production_smoke_result
    .decision
)

print(
    "Diagnosis:",
    production_smoke_result
    .diagnosis
)

print(
    "Retrieval relevant:",
    production_smoke_result
    .retrieval_relevant
)

print(
    "Top-1 relevance:",
    production_smoke_result
    .retrieval_top1_score
)

print(
    "Confidence:",
    production_smoke_result
    .confidence
)

print(
    "Trace:",
    production_smoke_result
    .trace
)

Decision: approved
Diagnosis: metric_migration
Retrieval relevant: True
Top-1 relevance: 0.9938342571258545
Confidence: 0.95
Trace: ['evidence_retrieval_agent', 'metric_investigation_agent', 'verification_reporting_agent']


In [64]:
assert (
    production_smoke_result
    .retrieval_relevant
    is True
)

assert (
    production_smoke_result
    .retrieval_top1_score
    is not None
)

assert (
    production_smoke_result
    .retrieval_top1_score
    >= relevance_config
       .top1_rerank_threshold
)

assert (
    production_smoke_result
    .decision
    == "approved"
)


print(
    "🔥 FINAL PRODUCTION RUNTIME "
    "EQUIVALENCE VALIDATED."
)

🔥 FINAL PRODUCTION RUNTIME EQUIVALENCE VALIDATED.


---

## Phase 9.3 — Formal Scoring

Production answers are scored against the quarantined expected answers.

Two independent evaluation dimensions are measured:

### Required-Fact Coverage

Each expected fact is tested against the generated answer using a separate
Natural Language Inference model.

The evaluator determines whether the generated answer semantically entails
each required fact.

### Classification Accuracy

Questions with an expected conflict classification are compared against the
allowed production diagnosis taxonomy.

Questions without a ground-truth classification are excluded from the
classification denominator.

No production answers are regenerated in this phase.

In [65]:
assert len(
    evaluation_results
) == 10


evaluation_errors = [
    result
    for result
    in evaluation_results
    if "error" in result
]


assert not evaluation_errors, (
    f"Evaluation contains errors: "
    f"{evaluation_errors}"
)


assert all(
    result.get(
        "answer"
    )
    for result
    in evaluation_results
)


print(
    "✅ 10 existing production answers "
    "ready for scoring."
)

✅ 10 existing production answers ready for scoring.


In [66]:
result_lookup = {
    result["question_id"]:
        result
    for result
    in evaluation_results
}


assert (
    set(result_lookup)
    == {
        case["question_id"]
        for case
        in evaluation_cases
    }
)


print(
    "✅ Production results aligned "
    "with ground truth."
)

✅ Production results aligned with ground truth.


In [67]:
EXPECTED_CLASSIFICATION_TO_DIAGNOSES = {

    "stale_version_conflict": {
        "version_mismatch",
        "stale_definition",
        "metric_migration",
    },

    "stale_semantic_definition": {
        "stale_definition",
        "version_mismatch",
    },

    "expected_semantic_difference": {
        "intentional_semantic_difference",
    },

    "expected_metric_migration": {
        "metric_migration",
    },
}


classification_rows = []


for case in evaluation_cases:

    expected = (
        case[
            "expected_classification"
        ]
    )

    predicted = (
        result_lookup[
            case[
                "question_id"
            ]
        ][
            "predicted_diagnosis"
        ]
    )


    if expected is None:

        correct = None

        allowed = None

    else:

        allowed = (
            EXPECTED_CLASSIFICATION_TO_DIAGNOSES[
                expected
            ]
        )

        correct = (
            predicted
            in allowed
        )


    classification_rows.append(
        {
            "question_id":
                case[
                    "question_id"
                ],

            "category":
                case[
                    "category"
                ],

            "expected_classification":
                expected,

            "predicted_diagnosis":
                predicted,

            "allowed_diagnoses":
                (
                    sorted(
                        allowed
                    )
                    if allowed
                    else None
                ),

            "classification_correct":
                correct,
        }
    )


classification_df = (
    pd.DataFrame(
        classification_rows
    )
)


classification_df

,question_id,category,expected_classification,predicted_diagnosis,allowed_diagnoses,classification_correct
0,Q001,conflict_detection,stale_version_conflict,metric_migration,"[metric_migration, stale_definition, version_m...",True
1,Q002,version_detection,None,metric_migration,None,None
2,Q003,staleness_detection,stale_version_conflict,version_mismatch,"[metric_migration, stale_definition, version_m...",True
3,Q004,semantic_conflict,stale_semantic_definition,intentional_semantic_difference,"[stale_definition, version_mismatch]",False
4,Q005,semantic_classification,expected_semantic_difference,intentional_semantic_difference,[intentional_semantic_difference],True
5,Q006,version_detection,None,metric_migration,None,None
6,Q007,metric_migration,expected_metric_migration,metric_migration,[metric_migration],True
7,Q008,lineage,None,metric_migration,None,None
8,Q009,impact_analysis,None,version_mismatch,None,None
9,Q010,version_detection,None,metric_migration,None,None


In [68]:
classified_df = (
    classification_df[
        classification_df[
            "classification_correct"
        ]
        .notna()
    ]
)


classification_accuracy = (
    classified_df[
        "classification_correct"
    ]
    .mean()
)


print(
    "Classified questions:",
    len(
        classified_df
    )
)

print(
    "Correct classifications:",
    int(
        classified_df[
            "classification_correct"
        ]
        .sum()
    )
)

print(
    "Classification accuracy:",
    round(
        classification_accuracy,
        4
    )
)

print(
    "Classification accuracy %:",
    round(
        classification_accuracy
        * 100,
        2
    )
)

Classified questions: 5
Correct classifications: 4
Classification accuracy: 0.8
Classification accuracy %: 80.0


In [69]:
%pip install -q sentencepiece

In [70]:
import torch

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)


NLI_MODEL_NAME = (
    "cross-encoder/"
    "nli-deberta-v3-base"
)


nli_device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


nli_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        NLI_MODEL_NAME
    )
)


nli_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        NLI_MODEL_NAME
    )
)


nli_model = (
    nli_model.to(
        nli_device
    )
)


nli_model.eval()


print(
    "✅ Independent NLI evaluator loaded."
)

print(
    "Device:",
    nli_device
)

print(
    "Labels:",
    nli_model.config.id2label
)

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

✅ Independent NLI evaluator loaded.
Device: cpu
Labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}


In [71]:
id2label = {
    int(index):
        str(label).lower()

    for index, label
    in nli_model
       .config
       .id2label
       .items()
}


entailment_indices = [
    index
    for index, label
    in id2label.items()
    if "entail" in label
]


assert (
    len(
        entailment_indices
    )
    == 1
), (
    "Could not uniquely identify "
    "the entailment label: "
    f"{id2label}"
)


ENTAILMENT_INDEX = (
    entailment_indices[0]
)


print(
    "✅ Entailment label:",
    id2label[
        ENTAILMENT_INDEX
    ]
)

print(
    "Entailment index:",
    ENTAILMENT_INDEX
)

✅ Entailment label: entailment
Entailment index: 1


In [72]:
def build_evaluation_premise(
    result: dict,
) -> str:

    sections = []


    answer = (
        result.get(
            "answer"
        )
        or ""
    )

    if answer:
        sections.append(
            answer
        )


    findings = (
        result.get(
            "key_findings"
        )
        or []
    )


    if findings:

        sections.append(
            "\n".join(
                str(item)
                for item
                in findings
            )
        )


    return (
        "\n".join(
            sections
        )
    )

In [73]:
fact_pairs = []


for case in evaluation_cases:

    result = (
        result_lookup[
            case[
                "question_id"
            ]
        ]
    )


    premise = (
        build_evaluation_premise(
            result
        )
    )


    for fact_index, fact in enumerate(
        case[
            "required_facts"
        ],
        start=1,
    ):

        fact_pairs.append(
            {
                "question_id":
                    case[
                        "question_id"
                    ],

                "category":
                    case[
                        "category"
                    ],

                "fact_index":
                    fact_index,

                "required_fact":
                    fact,

                "premise":
                    premise,
            }
        )


print(
    "✅ Required-fact pairs:",
    len(
        fact_pairs
    )
)

✅ Required-fact pairs: 35


In [74]:
import torch.nn.functional as F


BATCH_SIZE = 8


fact_scores = []


for start in range(
    0,
    len(fact_pairs),
    BATCH_SIZE,
):

    batch = (
        fact_pairs[
            start:
            start + BATCH_SIZE
        ]
    )


    premises = [
        item[
            "premise"
        ]
        for item
        in batch
    ]


    hypotheses = [
        item[
            "required_fact"
        ]
        for item
        in batch
    ]


    encoded = (
        nli_tokenizer(
            premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
    )


    encoded = {
        key:
            value.to(
                nli_device
            )

        for key, value
        in encoded.items()
    }


    with torch.no_grad():

        logits = (
            nli_model(
                **encoded
            )
            .logits
        )


    probabilities = (
        F.softmax(
            logits,
            dim=-1
        )
    )


    predicted_indices = (
        probabilities
        .argmax(
            dim=-1
        )
        .detach()
        .cpu()
        .tolist()
    )


    entailment_probs = (
        probabilities[
            :,
            ENTAILMENT_INDEX
        ]
        .detach()
        .cpu()
        .tolist()
    )


    for item, predicted_index, entailment_prob in zip(
        batch,
        predicted_indices,
        entailment_probs,
    ):

        fact_scores.append(
            {
                **item,

                "predicted_label":
                    id2label[
                        predicted_index
                    ],

                "entailment_probability":
                    float(
                        entailment_prob
                    ),

                "fact_entailed":
                    (
                        predicted_index
                        == ENTAILMENT_INDEX
                    ),
            }
        )


print(
    "🔥 Independent fact scoring complete."
)

🔥 Independent fact scoring complete.


In [75]:
fact_scoring_df = (
    pd.DataFrame(
        fact_scores
    )
)


fact_scoring_df[
    [
        "question_id",
        "fact_index",
        "required_fact",
        "predicted_label",
        "entailment_probability",
        "fact_entailed",
    ]
]

,question_id,fact_index,required_fact,predicted_label,entailment_probability,fact_entailed
0,Q001,1,Finance uses Net Revenue v3.,entailment,0.985307,True
1,Q001,2,Executive uses Net Revenue v2.,entailment,0.988090,True
2,Q001,3,Net Revenue v3 became effective on 2026-04-01.,entailment,0.876783,True
3,Q001,4,Version 3 deducts posted chargebacks while Ver...,entailment,0.993745,True
4,Q002,1,The current Net Revenue version is v3.,entailment,0.988175,True
5,Q002,2,It became effective on 2026-04-01.,neutral,0.006145,False
6,Q002,3,"The definition subtracts discounts, completed ...",neutral,0.000342,False
7,Q003,1,No.,contradiction,0.009075,False
8,Q003,2,The Executive KPI Dashboard uses Net Revenue v2.,contradiction,0.118729,False
9,Q003,3,The current enterprise version is v3.,contradiction,0.001041,False


In [76]:
question_fact_df = (
    fact_scoring_df
    .groupby(
        "question_id",
        as_index=False,
    )
    .agg(
        required_facts=(
            "fact_entailed",
            "size",
        ),

        facts_entailed=(
            "fact_entailed",
            "sum",
        ),

        mean_entailment_probability=(
            "entailment_probability",
            "mean",
        ),
    )
)


question_fact_df[
    "fact_coverage"
] = (
    question_fact_df[
        "facts_entailed"
    ]
    /
    question_fact_df[
        "required_facts"
    ]
)


question_fact_df[
    "all_required_facts_covered"
] = (
    question_fact_df[
        "facts_entailed"
    ]
    ==
    question_fact_df[
        "required_facts"
    ]
)


question_fact_df

,question_id,required_facts,facts_entailed,mean_entailment_probability,fact_coverage,all_required_facts_covered
0,Q001,4,4,0.960981,1.000000,True
1,Q002,3,1,0.331554,0.333333,False
2,Q003,3,0,0.042948,0.000000,False
3,Q004,3,0,0.001484,0.000000,False
4,Q005,4,3,0.731449,0.750000,False
5,Q006,3,0,0.157687,0.000000,False
6,Q007,4,3,0.752143,0.750000,False
7,Q008,4,0,0.121999,0.000000,False
8,Q009,3,2,0.663469,0.666667,False
9,Q010,4,4,0.996299,1.000000,True


In [77]:
total_required_facts = (
    len(
        fact_scoring_df
    )
)


total_entailed_facts = int(
    fact_scoring_df[
        "fact_entailed"
    ]
    .sum()
)


required_fact_accuracy = (
    total_entailed_facts
    /
    total_required_facts
)


full_question_accuracy = (
    question_fact_df[
        "all_required_facts_covered"
    ]
    .mean()
)


print(
    "Total required facts:",
    total_required_facts
)

print(
    "Entailed facts:",
    total_entailed_facts
)

print(
    "Required-fact accuracy:",
    round(
        required_fact_accuracy
        * 100,
        2
    ),
    "%"
)

print(
    "Questions with ALL facts covered:",
    int(
        question_fact_df[
            "all_required_facts_covered"
        ]
        .sum()
    ),
    "/",
    len(
        question_fact_df
    )
)

print(
    "Full-question factual accuracy:",
    round(
        full_question_accuracy
        * 100,
        2
    ),
    "%"
)

Total required facts: 35
Entailed facts: 17
Required-fact accuracy: 48.57 %
Questions with ALL facts covered: 2 / 10
Full-question factual accuracy: 20.0 %


In [78]:
missed_facts_df = (
    fact_scoring_df[
        fact_scoring_df[
            "fact_entailed"
        ]
        == False
    ][
        [
            "question_id",
            "category",
            "required_fact",
            "predicted_label",
            "entailment_probability",
        ]
    ]
)


if missed_facts_df.empty:

    print(
        "🔥 No required facts missed."
    )

else:

    print(
        "⚠️ Missed required facts:"
    )

    display(
        missed_facts_df
    )

⚠️ Missed required facts:


,question_id,category,required_fact,predicted_label,entailment_probability
5,Q002,version_detection,It became effective on 2026-04-01.,neutral,0.006145
6,Q002,version_detection,"The definition subtracts discounts, completed ...",neutral,0.000342
7,Q003,staleness_detection,No.,contradiction,0.009075
8,Q003,staleness_detection,The Executive KPI Dashboard uses Net Revenue v2.,contradiction,0.118729
9,Q003,staleness_detection,The current enterprise version is v3.,contradiction,0.001041
10,Q004,semantic_conflict,Growth uses Active Customers v1.,neutral,0.000808
11,Q004,semantic_conflict,Version 1 counts identified customers with rec...,neutral,0.001723
12,Q004,semantic_conflict,The enterprise v2 definition requires at least...,neutral,0.001922
13,Q005,semantic_classification,No underlying data pipeline failure was identi...,neutral,0.120197
17,Q006,version_detection,The current Total Orders version is v2.,neutral,0.001117


In [79]:
successful_runs = [
    result
    for result
    in evaluation_results
    if "error" not in result
]


retrieval_acceptance_rate = (
    sum(
        result.get(
            "retrieval_relevant"
        )
        is True

        for result
        in successful_runs
    )
    /
    len(
        successful_runs
    )
)


source_presence_rate = (
    sum(
        result.get(
            "source_count",
            0
        )
        > 0

        for result
        in successful_runs
    )
    /
    len(
        successful_runs
    )
)


mean_confidence = (
    sum(
        result[
            "confidence"
        ]
        for result
        in successful_runs
    )
    /
    len(
        successful_runs
    )
)


revision_rate = (
    sum(
        result[
            "revision_count"
        ]
        > 0

        for result
        in successful_runs
    )
    /
    len(
        successful_runs
    )
)


mean_latency = (
    sum(
        result[
            "latency_seconds"
        ]
        for result
        in successful_runs
    )
    /
    len(
        successful_runs
    )
)


total_llm_calls = sum(
    result[
        "estimated_llm_calls"
    ]
    for result
    in successful_runs
)


formal_metrics = {

    "questions_evaluated":
        len(
            successful_runs
        ),

    "execution_success_rate":
        len(
            successful_runs
        )
        / len(
            evaluation_cases
        ),

    "required_fact_accuracy":
        required_fact_accuracy,

    "full_question_fact_accuracy":
        full_question_accuracy,

    "classification_accuracy":
        classification_accuracy,

    "supported_retrieval_acceptance":
        retrieval_acceptance_rate,

    "source_presence_rate":
        source_presence_rate,

    "mean_confidence":
        mean_confidence,

    "revision_rate":
        revision_rate,

    "mean_latency_seconds":
        mean_latency,

    "total_llm_calls":
        total_llm_calls,
}


for metric, value in (
    formal_metrics.items()
):

    print(
        metric,
        "=",
        round(
            value,
            4
        )
        if isinstance(
            value,
            float,
        )
        else value
    )

questions_evaluated = 10
execution_success_rate = 1.0
required_fact_accuracy = 0.4857
full_question_fact_accuracy = 0.2
classification_accuracy = 0.8
supported_retrieval_acceptance = 1.0
source_presence_rate = 1.0
mean_confidence = 0.968
revision_rate = 0.0
mean_latency_seconds = 65.1272
total_llm_calls = 20


---

## Phase 9.3B — Required-Fact Failure Audit

The first automated NLI evaluation reported lower factual coverage than
expected.

Before treating those scores as product accuracy, missed facts are audited
against the actual generated answers to distinguish:

- genuine answer omissions
- incorrect answers
- overly strict NLI judgments
- paraphrase sensitivity
- compound-fact evaluation failures

No production answers are regenerated during this audit.

In [80]:
missed_fact_audit_rows = []


for _, row in missed_facts_df.iterrows():

    question_id = (
        row["question_id"]
    )

    production_result = (
        result_lookup[
            question_id
        ]
    )

    missed_fact_audit_rows.append(
        {
            "question_id":
                question_id,

            "category":
                row["category"],

            "required_fact":
                row[
                    "required_fact"
                ],

            "nli_label":
                row[
                    "predicted_label"
                ],

            "entailment_probability":
                row[
                    "entailment_probability"
                ],

            "production_answer":
                production_result[
                    "answer"
                ],

            "key_findings":
                production_result[
                    "key_findings"
                ],
        }
    )


missed_fact_audit_df = (
    pd.DataFrame(
        missed_fact_audit_rows
    )
)


display(
    missed_fact_audit_df
)

,question_id,category,required_fact,nli_label,entailment_probability,production_answer,key_findings
0,Q002,version_detection,It became effective on 2026-04-01.,neutral,0.006145,The current approved and active version of Net...,[Evidence E4 confirms that Net Revenue version...
1,Q002,version_detection,"The definition subtracts discounts, completed ...",neutral,0.000342,The current approved and active version of Net...,[Evidence E4 confirms that Net Revenue version...
2,Q003,staleness_detection,No.,contradiction,0.009075,"No, the Executive KPI Dashboard is not using t...",[Finance Analytics updated the authoritative e...
3,Q003,staleness_detection,The Executive KPI Dashboard uses Net Revenue v2.,contradiction,0.118729,"No, the Executive KPI Dashboard is not using t...",[Finance Analytics updated the authoritative e...
4,Q003,staleness_detection,The current enterprise version is v3.,contradiction,0.001041,"No, the Executive KPI Dashboard is not using t...",[Finance Analytics updated the authoritative e...
5,Q004,semantic_conflict,Growth uses Active Customers v1.,neutral,0.000808,The Growth & Marketing Dashboard reports more ...,[The Growth & Marketing Dashboard counts ident...
6,Q004,semantic_conflict,Version 1 counts identified customers with rec...,neutral,0.001723,The Growth & Marketing Dashboard reports more ...,[The Growth & Marketing Dashboard counts ident...
7,Q004,semantic_conflict,The enterprise v2 definition requires at least...,neutral,0.001922,The Growth & Marketing Dashboard reports more ...,[The Growth & Marketing Dashboard counts ident...
8,Q005,semantic_classification,No underlying data pipeline failure was identi...,neutral,0.120197,"No, the Total Orders disagreement between Oper...",[Analyst notes confirm that Operations and Fin...
9,Q006,version_detection,The current Total Orders version is v2.,neutral,0.001117,The current authoritative enterprise definitio...,[Under the current authoritative enterprise bu...


In [81]:
fact_failure_summary_df = (
    fact_scoring_df
    .groupby(
        [
            "question_id",
            "category",
        ],
        as_index=False,
    )
    .agg(
        total_required_facts=(
            "fact_entailed",
            "size",
        ),

        facts_covered=(
            "fact_entailed",
            "sum",
        ),
    )
)


fact_failure_summary_df[
    "facts_missed"
] = (
    fact_failure_summary_df[
        "total_required_facts"
    ]
    -
    fact_failure_summary_df[
        "facts_covered"
    ]
)


fact_failure_summary_df[
    "coverage_percent"
] = (
    fact_failure_summary_df[
        "facts_covered"
    ]
    /
    fact_failure_summary_df[
        "total_required_facts"
    ]
    * 100
)


display(
    fact_failure_summary_df
)

,question_id,category,total_required_facts,facts_covered,facts_missed,coverage_percent
0,Q001,conflict_detection,4,4,0,100.000000
1,Q002,version_detection,3,1,2,33.333333
2,Q003,staleness_detection,3,0,3,0.000000
3,Q004,semantic_conflict,3,0,3,0.000000
4,Q005,semantic_classification,4,3,1,75.000000
5,Q006,version_detection,3,0,3,0.000000
6,Q007,metric_migration,4,3,1,75.000000
7,Q008,lineage,4,0,4,0.000000
8,Q009,impact_analysis,3,2,1,66.666667
9,Q010,version_detection,4,4,0,100.000000


In [82]:
classification_failures_df = (
    classified_df[
        classified_df[
            "classification_correct"
        ]
        == False
    ]
)


display(
    classification_failures_df
)

,question_id,category,expected_classification,predicted_diagnosis,allowed_diagnoses,classification_correct
3,Q004,semantic_conflict,stale_semantic_definition,intentional_semantic_difference,"[stale_definition, version_mismatch]",False


---

## Phase 9.3C — Ground-Truth Fact Audit

The initial Natural Language Inference evaluator produced demonstrable false
negatives, including classifying explicitly supported negated facts as
contradictions.

NLI scores are therefore retained only as an auxiliary evaluator diagnostic.

The primary factual evaluation uses explicit ground-truth fact adjudication.

Each required fact is evaluated independently as:

- `supported` — the generated answer clearly contains the fact
- `missing` — the answer does not provide the required fact
- `incorrect` — the answer states a conflicting fact

This avoids treating evaluator-model errors as MetricGuard errors.

In [83]:
fact_audit_rows = []


for case in evaluation_cases:

    result = (
        result_lookup[
            case["question_id"]
        ]
    )

    answer_text = (
        result.get("answer", "")
    )

    key_findings = (
        result.get(
            "key_findings",
            [],
        )
        or []
    )

    full_output = (
        answer_text
        + "\n\n"
        + "\n".join(
            str(item)
            for item
            in key_findings
        )
    )


    for fact_index, fact in enumerate(
        case["required_facts"],
        start=1,
    ):

        fact_audit_rows.append(
            {
                "question_id":
                    case[
                        "question_id"
                    ],

                "category":
                    case[
                        "category"
                    ],

                "fact_index":
                    fact_index,

                "required_fact":
                    fact,

                "production_output":
                    full_output,

                "audit_status":
                    None,

                "audit_note":
                    None,
            }
        )


manual_fact_audit_df = (
    pd.DataFrame(
        fact_audit_rows
    )
)


print(
    "Required facts to audit:",
    len(
        manual_fact_audit_df
    )
)

display(
    manual_fact_audit_df[
        [
            "question_id",
            "fact_index",
            "required_fact",
        ]
    ]
)

Required facts to audit: 35


,question_id,fact_index,required_fact
0,Q001,1,Finance uses Net Revenue v3.
1,Q001,2,Executive uses Net Revenue v2.
2,Q001,3,Net Revenue v3 became effective on 2026-04-01.
3,Q001,4,Version 3 deducts posted chargebacks while Ver...
4,Q002,1,The current Net Revenue version is v3.
5,Q002,2,It became effective on 2026-04-01.
6,Q002,3,"The definition subtracts discounts, completed ..."
7,Q003,1,No.
8,Q003,2,The Executive KPI Dashboard uses Net Revenue v2.
9,Q003,3,The current enterprise version is v3.


In [84]:
def show_fact_audit(
    question_id: str,
):

    case = next(
        item
        for item
        in evaluation_cases
        if item[
            "question_id"
        ] == question_id
    )

    result = (
        result_lookup[
            question_id
        ]
    )


    print(
        "=" * 90
    )

    print(
        question_id,
        "-",
        case[
            "category"
        ]
    )

    print(
        "\nQUESTION:\n"
    )

    print(
        case[
            "question"
        ]
    )

    print(
        "\nPRODUCTION ANSWER:\n"
    )

    print(
        result[
            "answer"
        ]
    )

    print(
        "\nKEY FINDINGS:\n"
    )

    for finding in (
        result.get(
            "key_findings",
            []
        )
    ):

        print(
            "-",
            finding
        )


    print(
        "\nREQUIRED FACTS:\n"
    )

    for index, fact in enumerate(
        case[
            "required_facts"
        ],
        start=1,
    ):

        print(
            f"{index}.",
            fact
        )

In [85]:
for question_id in sorted(
    result_lookup
):

    show_fact_audit(
        question_id
    )

    print(
        "\n\n"
    )

Q001 - conflict_detection

QUESTION:

Why does the Executive KPI Dashboard report different Net Revenue from the Finance Revenue Dashboard after April 1, 2026?

PRODUCTION ANSWER:

The discrepancy occurs because Northstar Commerce updated the official enterprise Net Revenue definition to Version 3 effective April 1, 2026, which introduces the deduction of posted chargebacks. The Finance Revenue Dashboard and Finance Daily mart have successfully migrated to v3, whereas the Executive KPI Dashboard remains on Version 2 (which does not deduct posted chargebacks). Consequently, the Executive KPI Dashboard reports higher Net Revenue figures for reporting dates with chargeback activity.

KEY FINDINGS:

- Effective April 1, 2026, the authoritative enterprise Net Revenue definition was updated to Version 3 to deduct posted chargebacks [E5].
- Finance Analytics completed the migration of the Finance Revenue Dashboard and Finance Daily mart to Net Revenue v3 [E1, E3].
- The Executive KPI Dashboar

# Recording reviewed factual adjudications

## Phase 9.3C — Human-Verified Ground-Truth Audit

The NLI evaluator produced multiple demonstrable false negatives.

NLI entailment is therefore retained as an auxiliary evaluator diagnostic,
while NLI-negative required facts are manually adjudicated against the actual
production outputs.

Each required fact is labelled as:

- `supported`
- `missing`
- `incorrect`
- `review_required`

Existing NLI-positive facts are provisionally treated as supported unless a
manual review overrides them.

In [86]:
# Recording manually reviewed NLI-negative facts.

manual_fact_overrides = {

    # -----------------------------------------------------
    # Q002 — Net Revenue current version
    # -----------------------------------------------------

    ("Q002", 2): {
        "status": "supported",
        "note": (
            "The answer explicitly states that v3 "
            "became effective on 2026-04-01."
        ),
    },

    ("Q002", 3): {
        "status": "missing",
        "note": (
            "The answer mentions chargebacks but does "
            "not provide the complete v3 formula with "
            "discounts, completed refunds, and "
            "posted chargebacks."
        ),
    },


    # -----------------------------------------------------
    # Q003 — Executive dashboard staleness
    # -----------------------------------------------------

    ("Q003", 1): {
        "status": "supported",
        "note": (
            "The answer explicitly begins with 'No'."
        ),
    },

    ("Q003", 2): {
        "status": "supported",
        "note": (
            "The answer explicitly states that the "
            "Executive KPI Dashboard remains on v2."
        ),
    },

    ("Q003", 3): {
        "status": "supported",
        "note": (
            "The answer explicitly identifies v3 as "
            "the authoritative enterprise definition."
        ),
    },


    # -----------------------------------------------------
    # Q004 — Active Customers semantic conflict
    # -----------------------------------------------------

    ("Q004", 1): {
        "status": "missing",
        "note": (
            "The answer describes the Growth dashboard's "
            "older digital-activity semantics but never "
            "explicitly identifies them as Active "
            "Customers v1."
        ),
    },

    ("Q004", 2): {
        "status": "supported",
        "note": (
            "The answer explicitly describes identified "
            "customers with recent digital activity."
        ),
    },

    ("Q004", 3): {
        "status": "supported",
        "note": (
            "The answer explicitly states that enterprise "
            "v2 requires at least one successfully paid "
            "order in the previous 30 days."
        ),
    },


    # -----------------------------------------------------
    # Q006 — Total Orders current definition
    # -----------------------------------------------------

    ("Q006", 1): {
        "status": "supported",
        "note": (
            "The answer explicitly identifies the "
            "current Total Orders definition as v2."
        ),
    },

    ("Q006", 2): {
        "status": "supported",
        "note": (
            "The answer explicitly defines Total Orders "
            "as orders with a successful payment."
        ),
    },

    ("Q006", 3): {
        "status": "supported",
        "note": (
            "The key findings explicitly state that the "
            "definition became effective on "
            "2025-12-01."
        ),
    },


    # -----------------------------------------------------
    # Q008 — Net Revenue upstream lineage
    # -----------------------------------------------------

    ("Q008", 1): {
        "status": "supported",
        "note": (
            "The answer explicitly identifies "
            "mart_finance_daily as the dashboard's "
            "direct upstream node."
        ),
    },

    ("Q008", 2): {
        "status": "missing",
        "note": (
            "The answer does not state that "
            "mart_finance_daily depends on fct_orders."
        ),
    },

    ("Q008", 3): {
        "status": "missing",
        "note": (
            "The answer does not identify stg_orders, "
            "stg_payments, and stg_refunds as upstream "
            "dependencies of fct_orders."
        ),
    },

    ("Q008", 4): {
        "status": "missing",
        "note": (
            "The answer does not continue lineage to "
            "raw_orders, raw_payments, and raw_refunds."
        ),
    },
}


print(
    "✅ Manual fact adjudications recorded:",
    len(manual_fact_overrides)
)

✅ Manual fact adjudications recorded: 15


In [87]:
# Combining automated entailment with manual adjudication.

hybrid_fact_rows = []


for _, row in fact_scoring_df.iterrows():

    key = (
        row["question_id"],
        int(row["fact_index"]),
    )

    override = (
        manual_fact_overrides.get(
            key
        )
    )


    if override is not None:

        final_status = (
            override["status"]
        )

        audit_source = (
            "manual_review"
        )

        audit_note = (
            override["note"]
        )


    elif bool(
        row["fact_entailed"]
    ):

        final_status = (
            "supported"
        )

        audit_source = (
            "nli_entailment"
        )

        audit_note = (
            "NLI evaluator classified the "
            "required fact as entailed."
        )


    else:

        final_status = (
            "review_required"
        )

        audit_source = (
            "pending_manual_review"
        )

        audit_note = (
            "NLI rejected this fact and it has "
            "not yet been manually adjudicated."
        )


    hybrid_fact_rows.append(
        {
            "question_id":
                row["question_id"],

            "category":
                row["category"],

            "fact_index":
                int(
                    row["fact_index"]
                ),

            "required_fact":
                row["required_fact"],

            "nli_label":
                row["predicted_label"],

            "nli_entailment_probability":
                row[
                    "entailment_probability"
                ],

            "final_status":
                final_status,

            "audit_source":
                audit_source,

            "audit_note":
                audit_note,
        }
    )


hybrid_fact_audit_df = (
    pd.DataFrame(
        hybrid_fact_rows
    )
)


display(
    hybrid_fact_audit_df[
        [
            "question_id",
            "fact_index",
            "required_fact",
            "final_status",
            "audit_source",
        ]
    ]
)

,question_id,fact_index,required_fact,final_status,audit_source
0,Q001,1,Finance uses Net Revenue v3.,supported,nli_entailment
1,Q001,2,Executive uses Net Revenue v2.,supported,nli_entailment
2,Q001,3,Net Revenue v3 became effective on 2026-04-01.,supported,nli_entailment
3,Q001,4,Version 3 deducts posted chargebacks while Ver...,supported,nli_entailment
4,Q002,1,The current Net Revenue version is v3.,supported,nli_entailment
5,Q002,2,It became effective on 2026-04-01.,supported,manual_review
6,Q002,3,"The definition subtracts discounts, completed ...",missing,manual_review
7,Q003,1,No.,supported,manual_review
8,Q003,2,The Executive KPI Dashboard uses Net Revenue v2.,supported,manual_review
9,Q003,3,The current enterprise version is v3.,supported,manual_review


In [88]:
# Isolating facts still requiring manual review.

remaining_fact_reviews_df = (
    hybrid_fact_audit_df[
        hybrid_fact_audit_df[
            "final_status"
        ]
        == "review_required"
    ][
        [
            "question_id",
            "category",
            "fact_index",
            "required_fact",
            "nli_label",
            "nli_entailment_probability",
        ]
    ]
)


display(
    remaining_fact_reviews_df
)


print(
    "Remaining manual reviews:",
    len(
        remaining_fact_reviews_df
    )
)

,question_id,category,fact_index,required_fact,nli_label,nli_entailment_probability
13,Q005,semantic_classification,1,No underlying data pipeline failure was identi...,neutral,0.120197
23,Q007,metric_migration,4,The change was expected and not a data defect.,neutral,0.028883
29,Q009,impact_analysis,2,Its upstream mart is mart_executive_daily.,neutral,0.001266


Remaining manual reviews: 3


In [89]:
# Calculating the current audited factual-score bounds.

status_counts = (
    hybrid_fact_audit_df[
        "final_status"
    ]
    .value_counts()
)


supported_count = int(
    status_counts.get(
        "supported",
        0,
    )
)

missing_count = int(
    status_counts.get(
        "missing",
        0,
    )
)

incorrect_count = int(
    status_counts.get(
        "incorrect",
        0,
    )
)

review_count = int(
    status_counts.get(
        "review_required",
        0,
    )
)


total_fact_count = (
    len(
        hybrid_fact_audit_df
    )
)


minimum_accuracy = (
    supported_count
    /
    total_fact_count
)


maximum_accuracy = (
    (
        supported_count
        + review_count
    )
    /
    total_fact_count
)


print(
    "Total facts:",
    total_fact_count
)

print(
    "Supported:",
    supported_count
)

print(
    "Missing:",
    missing_count
)

print(
    "Incorrect:",
    incorrect_count
)

print(
    "Pending review:",
    review_count
)


print(
    "\nCurrent factual accuracy range:"
)


print(
    round(
        minimum_accuracy
        * 100,
        2,
    ),
    "%",
    "to",
    round(
        maximum_accuracy
        * 100,
        2,
    ),
    "%",
)

Total facts: 35
Supported: 27
Missing: 5
Incorrect: 0
Pending review: 3

Current factual accuracy range:
77.14 % to 85.71 %


In [90]:
# Printing unresolved production answers for manual adjudication.

for question_id in [
    "Q005",
    "Q007",
    "Q009",
]:

    result = (
        result_lookup[
            question_id
        ]
    )

    case = next(
        item
        for item
        in evaluation_cases
        if item[
            "question_id"
        ] == question_id
    )

    print(
        "=" * 90
    )

    print(
        question_id,
        "-",
        case[
            "category"
        ]
    )

    print(
        "\nQUESTION:\n"
    )

    print(
        case[
            "question"
        ]
    )

    print(
        "\nPRODUCTION ANSWER:\n"
    )

    print(
        result[
            "answer"
        ]
    )

    print(
        "\nKEY FINDINGS:\n"
    )

    for finding in (
        result.get(
            "key_findings",
            []
        )
    ):

        print(
            "-",
            finding
        )

    print(
        "\nREQUIRED FACTS:\n"
    )

    for index, fact in enumerate(
        case[
            "required_facts"
        ],
        start=1,
    ):

        print(
            f"{index}.",
            fact
        )

    print(
        "\n"
    )

Q005 - semantic_classification

QUESTION:

Is the Total Orders disagreement between Operations and Finance a data pipeline failure?

PRODUCTION ANSWER:

No, the Total Orders disagreement between Operations and Finance is not a data pipeline failure. It is an intentional semantic difference. Operations counts all placed orders (excluding cancellations) to track operational workload, whereas Finance tracks successfully paid orders following the enterprise KPI definition introduced in December 2025. Incident INC-003 and analyst documentation both confirm this discrepancy is expected and by design.

KEY FINDINGS:

- Analyst notes confirm that Operations and Finance intentionally apply different order-counting definitions to meet distinct business requirements (E1).
- Operations counts all valid placed orders except cancelled orders, while Finance reports on successfully paid orders in line with the enterprise KPI definition (E1, E5).
- Incident INC-003 formally investigated the discrepancy

# Finalizing factual adjudication

## Phase 9.3C — Final Human-Verified Fact Audit

The remaining NLI-negative facts have been reviewed against the original
production outputs.

Final adjudications:

- Q005 Fact 1 — `supported`
- Q007 Fact 4 — `supported`
- Q009 Fact 2 — `missing`

The primary factual benchmark now uses the completed ground-truth audit.

NLI entailment coverage is retained only as an auxiliary evaluator diagnostic.

In [91]:
# Recording final manual adjudications.

manual_fact_overrides.update(
    {
        ("Q005", 1): {
            "status": "supported",
            "note": (
                "The answer explicitly states that the "
                "disagreement is not a data pipeline "
                "failure, and INC-003 confirms that it "
                "is not a pipeline defect."
            ),
        },

        ("Q007", 4): {
            "status": "supported",
            "note": (
                "The answer explicitly describes the "
                "migration as intentional, planned, "
                "and expected rather than a defect."
            ),
        },

        ("Q009", 2): {
            "status": "missing",
            "note": (
                "The answer refers generally to the "
                "Executive pipeline but does not name "
                "mart_executive_daily as the upstream mart."
            ),
        },
    }
)


print(
    "✅ Total manual adjudications:",
    len(manual_fact_overrides)
)

✅ Total manual adjudications: 18


In [92]:
# Rebuilding the completed hybrid fact audit.

final_fact_rows = []


for _, row in fact_scoring_df.iterrows():

    key = (
        row["question_id"],
        int(row["fact_index"]),
    )

    override = (
        manual_fact_overrides.get(
            key
        )
    )


    if override is not None:

        final_status = (
            override["status"]
        )

        audit_source = (
            "manual_review"
        )

        audit_note = (
            override["note"]
        )


    elif bool(
        row["fact_entailed"]
    ):

        final_status = (
            "supported"
        )

        audit_source = (
            "nli_entailment"
        )

        audit_note = (
            "NLI evaluator classified the "
            "required fact as entailed."
        )


    else:

        final_status = (
            "review_required"
        )

        audit_source = (
            "pending_manual_review"
        )

        audit_note = (
            "Fact requires manual review."
        )


    final_fact_rows.append(
        {
            "question_id":
                row["question_id"],

            "category":
                row["category"],

            "fact_index":
                int(
                    row["fact_index"]
                ),

            "required_fact":
                row["required_fact"],

            "nli_label":
                row["predicted_label"],

            "nli_entailment_probability":
                row[
                    "entailment_probability"
                ],

            "final_status":
                final_status,

            "audit_source":
                audit_source,

            "audit_note":
                audit_note,
        }
    )


final_fact_audit_df = (
    pd.DataFrame(
        final_fact_rows
    )
)


assert (
    "review_required"
    not in
    final_fact_audit_df[
        "final_status"
    ].values
)


print(
    "🔥 All 35 required facts adjudicated."
)

🔥 All 35 required facts adjudicated.


In [93]:
# Calculating final ground-truth factual accuracy.

final_status_counts = (
    final_fact_audit_df[
        "final_status"
    ]
    .value_counts()
)


final_supported = int(
    final_status_counts.get(
        "supported",
        0,
    )
)

final_missing = int(
    final_status_counts.get(
        "missing",
        0,
    )
)

final_incorrect = int(
    final_status_counts.get(
        "incorrect",
        0,
    )
)

final_total_facts = (
    len(
        final_fact_audit_df
    )
)


final_fact_accuracy = (
    final_supported
    /
    final_total_facts
)


print(
    "Total required facts:",
    final_total_facts
)

print(
    "Supported:",
    final_supported
)

print(
    "Missing:",
    final_missing
)

print(
    "Incorrect:",
    final_incorrect
)

print(
    "\nGround-truth factual accuracy:",
    round(
        final_fact_accuracy
        * 100,
        2,
    ),
    "%"
)

Total required facts: 35
Supported: 29
Missing: 6
Incorrect: 0

Ground-truth factual accuracy: 82.86 %


In [94]:
# Calculating question-level factual completeness.

audited_question_fact_df = (
    final_fact_audit_df
    .assign(
        fact_supported=(
            final_fact_audit_df[
                "final_status"
            ]
            == "supported"
        )
    )
    .groupby(
        "question_id",
        as_index=False,
    )
    .agg(
        required_facts=(
            "fact_supported",
            "size",
        ),

        supported_facts=(
            "fact_supported",
            "sum",
        ),
    )
)


audited_question_fact_df[
    "all_facts_supported"
] = (
    audited_question_fact_df[
        "required_facts"
    ]
    ==
    audited_question_fact_df[
        "supported_facts"
    ]
)


audited_question_fact_df[
    "fact_coverage"
] = (
    audited_question_fact_df[
        "supported_facts"
    ]
    /
    audited_question_fact_df[
        "required_facts"
    ]
)


fully_correct_questions = int(
    audited_question_fact_df[
        "all_facts_supported"
    ]
    .sum()
)


full_question_accuracy = (
    fully_correct_questions
    /
    len(
        audited_question_fact_df
    )
)


display(
    audited_question_fact_df
)


print(
    "\nFully correct questions:",
    fully_correct_questions,
    "/",
    len(
        audited_question_fact_df
    )
)

print(
    "Full-question factual accuracy:",
    round(
        full_question_accuracy
        * 100,
        2,
    ),
    "%"
)

,question_id,required_facts,supported_facts,all_facts_supported,fact_coverage
0,Q001,4,4,True,1.000000
1,Q002,3,2,False,0.666667
2,Q003,3,3,True,1.000000
3,Q004,3,2,False,0.666667
4,Q005,4,4,True,1.000000
5,Q006,3,3,True,1.000000
6,Q007,4,4,True,1.000000
7,Q008,4,1,False,0.250000
8,Q009,3,2,False,0.666667
9,Q010,4,4,True,1.000000



Fully correct questions: 6 / 10
Full-question factual accuracy: 60.0 %


In [95]:
# Locking final answer-quality metrics.

answer_quality_metrics = {
    "questions_evaluated":
        10,

    "required_facts_evaluated":
        final_total_facts,

    "supported_required_facts":
        final_supported,

    "missing_required_facts":
        final_missing,

    "incorrect_required_facts":
        final_incorrect,

    "ground_truth_fact_accuracy":
        final_fact_accuracy,

    "full_question_accuracy":
        full_question_accuracy,

    "classification_accuracy":
        classification_accuracy,

    "nli_auxiliary_fact_accuracy":
        required_fact_accuracy,

    "execution_success_rate":
        1.0,

    "supported_retrieval_acceptance":
        1.0,

    "source_presence_rate":
        1.0,

    "mean_confidence":
        mean_confidence,

    "revision_rate":
        revision_rate,

    "mean_latency_seconds":
        mean_latency,

    "total_llm_calls":
        total_llm_calls,
}


for metric, value in (
    answer_quality_metrics.items()
):

    print(
        metric,
        "=",
        round(
            value,
            4,
        )
        if isinstance(
            value,
            float,
        )
        else value
    )

questions_evaluated = 10
required_facts_evaluated = 35
supported_required_facts = 29
missing_required_facts = 6
incorrect_required_facts = 0
ground_truth_fact_accuracy = 0.8286
full_question_accuracy = 0.6
classification_accuracy = 0.8
nli_auxiliary_fact_accuracy = 0.4857
execution_success_rate = 1.0
supported_retrieval_acceptance = 1.0
source_presence_rate = 1.0
mean_confidence = 0.968
revision_rate = 0.0
mean_latency_seconds = 65.1272
total_llm_calls = 20


# Evaluating unsupported-query safety

## Phase 9.4 — Unsupported and Out-of-Domain Evaluation

The production relevance gate is evaluated against questions that cannot be
answered from MetricGuard's analytics-governance knowledge base.

These questions deliberately target unrelated domains such as:

- facilities
- employee compensation
- real estate
- transportation costs
- HR policy
- warehouse equipment

Expected behavior:

1. retrieval executes
2. mandatory reranking executes
3. Top-1 relevance remains below `0.27`
4. evidence is rejected
5. Agent 2 is skipped
6. Agent 3 is skipped
7. Gemini is not called
8. the final application result becomes `insufficient_evidence`

In [96]:
# Defining unsupported safety-evaluation questions.

unsupported_evaluation_cases = [
    {
        "question_id": "U001",
        "category": "facilities",
        "question": (
            "How much electricity did the warehouse "
            "HVAC system consume last month?"
        ),
    },

    {
        "question_id": "U002",
        "category": "employee_compensation",
        "question": (
            "What is the average employee salary "
            "at Northstar Commerce?"
        ),
    },

    {
        "question_id": "U003",
        "category": "real_estate",
        "question": (
            "What is the monthly rent for "
            "Northstar Commerce offices?"
        ),
    },

    {
        "question_id": "U004",
        "category": "transportation_cost",
        "question": (
            "How much did delivery fuel cost "
            "Northstar Commerce last quarter?"
        ),
    },

    {
        "question_id": "U005",
        "category": "hr_policy",
        "question": (
            "What is Northstar Commerce's "
            "employee vacation policy?"
        ),
    },

    {
        "question_id": "U006",
        "category": "warehouse_equipment",
        "question": (
            "What are the dimensions of the "
            "warehouse storage shelves?"
        ),
    },
]


print(
    "✅ Unsupported cases:",
    len(
        unsupported_evaluation_cases
    )
)

✅ Unsupported cases: 6


In [97]:
# Auditing unsupported retrieval relevance before LLM execution.

unsupported_retrieval_rows = []


for case in unsupported_evaluation_cases:

    candidates = (
        production_retrieval
        .retrieve(
            case[
                "question"
            ]
        )
    )


    relevance_decision = (
        relevance_gate.assess(
            candidates
        )
    )


    unsupported_retrieval_rows.append(
        {
            "question_id":
                case[
                    "question_id"
                ],

            "category":
                case[
                    "category"
                ],

            "question":
                case[
                    "question"
                ],

            "top1_score":
                relevance_decision
                .top1_rerank_score,

            "threshold":
                relevance_decision
                .threshold,

            "is_relevant":
                relevance_decision
                .is_relevant,

            "reason":
                relevance_decision
                .reason,
        }
    )


unsupported_retrieval_df = (
    pd.DataFrame(
        unsupported_retrieval_rows
    )
)


display(
    unsupported_retrieval_df
)

,question_id,category,question,top1_score,threshold,is_relevant,reason
0,U001,facilities,How much electricity did the warehouse HVAC sy...,0.000012,0.27,False,Top reranked candidate did not pass the calibr...
1,U002,employee_compensation,What is the average employee salary at Northst...,0.010349,0.27,False,Top reranked candidate did not pass the calibr...
2,U003,real_estate,What is the monthly rent for Northstar Commerc...,0.006939,0.27,False,Top reranked candidate did not pass the calibr...
3,U004,transportation_cost,How much did delivery fuel cost Northstar Comm...,0.000248,0.27,False,Top reranked candidate did not pass the calibr...
4,U005,hr_policy,What is Northstar Commerce's employee vacation...,0.012042,0.27,False,Top reranked candidate did not pass the calibr...
5,U006,warehouse_equipment,What are the dimensions of the warehouse stora...,0.000012,0.27,False,Top reranked candidate did not pass the calibr...


In [98]:
# Validating rejection of every unsupported retrieval.

assert all(
    item[
        "is_relevant"
    ]
    is False

    for item
    in unsupported_retrieval_rows
), (
    "At least one unsupported question passed "
    "the relevance gate. Stopping before Gemini."
)


maximum_unsupported_score = max(
    item[
        "top1_score"
    ]

    for item
    in unsupported_retrieval_rows
)


print(
    "Maximum unsupported Top-1 score:",
    maximum_unsupported_score
)

print(
    "Production threshold:",
    relevance_config
    .top1_rerank_threshold
)

print(
    "\n🔥 ALL UNSUPPORTED QUERIES "
    "REJECTED BEFORE LLM EXECUTION."
)

Maximum unsupported Top-1 score: 0.012041553854942322
Production threshold: 0.27

🔥 ALL UNSUPPORTED QUERIES REJECTED BEFORE LLM EXECUTION.


In [99]:
# Clearing runtime cache before safety evaluation.

if (
    metricguard.cache
    is not None
):

    metricguard.cache.clear()


print(
    "✅ Safety evaluation cache cleared."
)

✅ Safety evaluation cache cleared.


In [100]:
# Running unsupported questions through the production service.

import time


unsupported_results = []


for case in unsupported_evaluation_cases:

    start_time = (
        time.perf_counter()
    )


    result = (
        metricguard.ask(
            case[
                "question"
            ]
        )
    )


    latency = (
        time.perf_counter()
        - start_time
    )


    llm_calls = (
        result.trace.count(
            "metric_investigation_agent"
        )
        +
        result.trace.count(
            "verification_reporting_agent"
        )
    )


    unsupported_results.append(
        {
            "question_id":
                case[
                    "question_id"
                ],

            "category":
                case[
                    "category"
                ],

            "status":
                result.status,

            "decision":
                result.decision,

            "diagnosis":
                result.diagnosis,

            "retrieval_relevant":
                result
                .retrieval_relevant,

            "top1_relevance":
                result
                .retrieval_top1_score,

            "llm_calls":
                llm_calls,

            "latency_seconds":
                latency,

            "trace":
                result.trace,
        }
    )


unsupported_results_df = (
    pd.DataFrame(
        unsupported_results
    )
)


display(
    unsupported_results_df
)

,question_id,category,status,decision,diagnosis,retrieval_relevant,top1_relevance,llm_calls,latency_seconds,trace
0,U001,facilities,insufficient_evidence,insufficient_evidence,insufficient_evidence,False,0.000012,0,3.505639,"[evidence_retrieval_agent, relevance_gate_reje..."
1,U002,employee_compensation,insufficient_evidence,insufficient_evidence,insufficient_evidence,False,0.010349,0,4.367696,"[evidence_retrieval_agent, relevance_gate_reje..."
2,U003,real_estate,insufficient_evidence,insufficient_evidence,insufficient_evidence,False,0.006939,0,2.618888,"[evidence_retrieval_agent, relevance_gate_reje..."
3,U004,transportation_cost,insufficient_evidence,insufficient_evidence,insufficient_evidence,False,0.000248,0,3.129783,"[evidence_retrieval_agent, relevance_gate_reje..."
4,U005,hr_policy,insufficient_evidence,insufficient_evidence,insufficient_evidence,False,0.012042,0,2.566628,"[evidence_retrieval_agent, relevance_gate_reje..."
5,U006,warehouse_equipment,insufficient_evidence,insufficient_evidence,insufficient_evidence,False,0.000012,0,4.523035,"[evidence_retrieval_agent, relevance_gate_reje..."


In [101]:
# Validating deterministic unsupported-query fallback.

expected_rejection_trace = [
    "evidence_retrieval_agent",
    "relevance_gate_rejected",
    "no_evidence_fallback",
]


for result in unsupported_results:

    assert (
        result[
            "status"
        ]
        == "insufficient_evidence"
    )

    assert (
        result[
            "decision"
        ]
        == "insufficient_evidence"
    )

    assert (
        result[
            "diagnosis"
        ]
        == "insufficient_evidence"
    )

    assert (
        result[
            "retrieval_relevant"
        ]
        is False
    )

    assert (
        result[
            "llm_calls"
        ]
        == 0
    )

    assert (
        result[
            "trace"
        ]
        == expected_rejection_trace
    )


print(
    "🔥 UNSUPPORTED SAFETY VALIDATION PASSED."
)

🔥 UNSUPPORTED SAFETY VALIDATION PASSED.


In [102]:
# Calculating unsupported-query safety metrics.

unsupported_case_count = (
    len(
        unsupported_results
    )
)


rejected_case_count = sum(
    result[
        "status"
    ]
    == "insufficient_evidence"

    for result
    in unsupported_results
)


zero_llm_case_count = sum(
    result[
        "llm_calls"
    ]
    == 0

    for result
    in unsupported_results
)


unsupported_rejection_rate = (
    rejected_case_count
    /
    unsupported_case_count
)


zero_llm_rate = (
    zero_llm_case_count
    /
    unsupported_case_count
)


false_acceptance_rate = (
    1
    -
    unsupported_rejection_rate
)


mean_unsupported_latency = (
    sum(
        result[
            "latency_seconds"
        ]

        for result
        in unsupported_results
    )
    /
    unsupported_case_count
)


safety_metrics = {
    "unsupported_questions":
        unsupported_case_count,

    "unsupported_rejection_rate":
        unsupported_rejection_rate,

    "false_acceptance_rate":
        false_acceptance_rate,

    "zero_llm_execution_rate":
        zero_llm_rate,

    "maximum_unsupported_top1":
        maximum_unsupported_score,

    "relevance_threshold":
        relevance_config
        .top1_rerank_threshold,

    "mean_unsupported_latency_seconds":
        mean_unsupported_latency,
}


for metric, value in (
    safety_metrics.items()
):

    print(
        metric,
        "=",
        round(
            value,
            4,
        )
        if isinstance(
            value,
            float,
        )
        else value
    )

unsupported_questions = 6
unsupported_rejection_rate = 1.0
false_acceptance_rate = 0.0
zero_llm_execution_rate = 1.0
maximum_unsupported_top1 = 0.012
relevance_threshold = 0.27
mean_unsupported_latency_seconds = 3.4519


In [103]:
# Validating empty-question input protection.

empty_question_rejected = False


try:

    metricguard.ask(
        "   "
    )


except ValueError:

    empty_question_rejected = True


assert (
    empty_question_rejected
    is True
)


print(
    "🔥 EMPTY QUESTION VALIDATION PASSED."
)

🔥 EMPTY QUESTION VALIDATION PASSED.


# Summarizing the formal evaluation

## Phase 9.5 — Final Evaluation Synthesis

The supported-question benchmark, human-verified factual audit,
classification evaluation, retrieval evaluation, and unsupported-query
safety evaluation are now combined into one formal MetricGuard benchmark.

Primary metrics use deterministic or human-verified ground truth.

The experimental NLI entailment score is retained separately as an
auxiliary evaluator diagnostic because manual auditing exposed substantial
false-negative behavior.

In [104]:
# Building the final evaluation benchmark.

final_evaluation_metrics = {

    # -----------------------------------------------------
    # DATASET
    # -----------------------------------------------------

    "supported_questions":
        10,

    "unsupported_questions":
        6,

    "required_facts":
        35,

    "classified_questions":
        len(
            classified_df
        ),


    # -----------------------------------------------------
    # EXECUTION
    # -----------------------------------------------------

    "execution_success_rate":
        1.0,


    # -----------------------------------------------------
    # ANSWER QUALITY
    # -----------------------------------------------------

    "ground_truth_fact_accuracy":
        final_fact_accuracy,

    "full_question_fact_accuracy":
        full_question_accuracy,

    "classification_accuracy":
        classification_accuracy,

    "supported_required_facts":
        final_supported,

    "missing_required_facts":
        final_missing,

    "incorrect_required_facts":
        final_incorrect,


    # -----------------------------------------------------
    # RETRIEVAL
    # -----------------------------------------------------

    "supported_retrieval_acceptance":
        1.0,

    "source_presence_rate":
        1.0,

    "relevance_threshold":
        relevance_config
        .top1_rerank_threshold,

    "maximum_unsupported_top1":
        maximum_unsupported_score,


    # -----------------------------------------------------
    # SAFETY
    # -----------------------------------------------------

    "unsupported_rejection_rate":
        unsupported_rejection_rate,

    "false_acceptance_rate":
        false_acceptance_rate,

    "zero_llm_unsupported_rate":
        zero_llm_rate,


    # -----------------------------------------------------
    # AGENT BEHAVIOR
    # -----------------------------------------------------

    "mean_confidence":
        mean_confidence,

    "revision_rate":
        revision_rate,

    "supported_llm_calls":
        total_llm_calls,


    # -----------------------------------------------------
    # LATENCY
    # -----------------------------------------------------

    "mean_supported_latency_seconds":
        mean_latency,

    "mean_unsupported_latency_seconds":
        mean_unsupported_latency,


    # -----------------------------------------------------
    # AUXILIARY EVALUATOR
    # -----------------------------------------------------

    "nli_auxiliary_fact_accuracy":
        required_fact_accuracy,
}


print(
    "🔥 Final evaluation benchmark assembled."
)

🔥 Final evaluation benchmark assembled.


In [105]:
# Creating the headline evaluation metrics table.

headline_metrics_df = pd.DataFrame(
    [
        {
            "metric":
                "Execution success",

            "value":
                final_evaluation_metrics[
                    "execution_success_rate"
                ]
                * 100,

            "unit":
                "%",
        },

        {
            "metric":
                "Human-audited factual coverage",

            "value":
                final_evaluation_metrics[
                    "ground_truth_fact_accuracy"
                ]
                * 100,

            "unit":
                "%",
        },

        {
            "metric":
                "Fully complete answers",

            "value":
                final_evaluation_metrics[
                    "full_question_fact_accuracy"
                ]
                * 100,

            "unit":
                "%",
        },

        {
            "metric":
                "Classification accuracy",

            "value":
                final_evaluation_metrics[
                    "classification_accuracy"
                ]
                * 100,

            "unit":
                "%",
        },

        {
            "metric":
                "Supported retrieval acceptance",

            "value":
                final_evaluation_metrics[
                    "supported_retrieval_acceptance"
                ]
                * 100,

            "unit":
                "%",
        },

        {
            "metric":
                "Source presence",

            "value":
                final_evaluation_metrics[
                    "source_presence_rate"
                ]
                * 100,

            "unit":
                "%",
        },

        {
            "metric":
                "Unsupported rejection",

            "value":
                final_evaluation_metrics[
                    "unsupported_rejection_rate"
                ]
                * 100,

            "unit":
                "%",
        },

        {
            "metric":
                "Unsupported false acceptance",

            "value":
                final_evaluation_metrics[
                    "false_acceptance_rate"
                ]
                * 100,

            "unit":
                "%",
        },

        {
            "metric":
                "Unsupported zero-LLM execution",

            "value":
                final_evaluation_metrics[
                    "zero_llm_unsupported_rate"
                ]
                * 100,

            "unit":
                "%",
        },

        {
            "metric":
                "Mean supported latency",

            "value":
                final_evaluation_metrics[
                    "mean_supported_latency_seconds"
                ],

            "unit":
                "seconds",
        },

        {
            "metric":
                "Mean unsupported latency",

            "value":
                final_evaluation_metrics[
                    "mean_unsupported_latency_seconds"
                ],

            "unit":
                "seconds",
        },
    ]
)


headline_metrics_df[
    "value"
] = (
    headline_metrics_df[
        "value"
    ]
    .round(2)
)


display(
    headline_metrics_df
)

,metric,value,unit
0,Execution success,100.00,%
1,Human-audited factual coverage,82.86,%
2,Fully complete answers,60.00,%
3,Classification accuracy,80.00,%
4,Supported retrieval acceptance,100.00,%
5,Source presence,100.00,%
6,Unsupported rejection,100.00,%
7,Unsupported false acceptance,0.00,%
8,Unsupported zero-LLM execution,100.00,%
9,Mean supported latency,65.13,seconds


In [106]:
# Creating the final missed-facts analysis.

final_missed_facts_df = (
    final_fact_audit_df[
        final_fact_audit_df[
            "final_status"
        ]
        != "supported"
    ][
        [
            "question_id",
            "category",
            "fact_index",
            "required_fact",
            "final_status",
            "audit_note",
        ]
    ]
    .reset_index(
        drop=True
    )
)


display(
    final_missed_facts_df
)


print(
    "Total unsupported required facts:",
    len(
        final_missed_facts_df
    )
)

,question_id,category,fact_index,required_fact,final_status,audit_note
0,Q002,version_detection,3,"The definition subtracts discounts, completed ...",missing,The answer mentions chargebacks but does not p...
1,Q004,semantic_conflict,1,Growth uses Active Customers v1.,missing,The answer describes the Growth dashboard's ol...
2,Q008,lineage,2,mart_finance_daily depends on fct_orders.,missing,The answer does not state that mart_finance_da...
3,Q008,lineage,3,"fct_orders depends on stg_orders, stg_payments...",missing,"The answer does not identify stg_orders, stg_p..."
4,Q008,lineage,4,Those staging models originate from raw_orders...,missing,The answer does not continue lineage to raw_or...
5,Q009,impact_analysis,2,Its upstream mart is mart_executive_daily.,missing,The answer refers generally to the Executive p...


Total unsupported required facts: 6


In [107]:
# Creating the final classification-failure analysis.

final_classification_failures_df = (
    classification_df[
        classification_df[
            "classification_correct"
        ]
        == False
    ]
    .reset_index(
        drop=True
    )
)


display(
    final_classification_failures_df
)


print(
    "Classification failures:",
    len(
        final_classification_failures_df
    )
)

,question_id,category,expected_classification,predicted_diagnosis,allowed_diagnoses,classification_correct
0,Q004,semantic_conflict,stale_semantic_definition,intentional_semantic_difference,"[stale_definition, version_mismatch]",False


Classification failures: 1


# Documenting observed benchmark limitations

## Evaluation Findings

### 1. Factual completeness

MetricGuard supported `29 / 35` required ground-truth facts.

Six required facts were omitted, while no manually audited required fact was
identified as explicitly incorrect.

The largest factual-completeness failure occurred on the full upstream
lineage question, where the answer stopped at `mart_finance_daily` instead
of traversing through fact, staging, and raw models.

### 2. Governance classification

Classification accuracy was `4 / 5`.

The main classification failure occurred for Active Customers.

MetricGuard correctly explained the semantic difference between the Growth
dashboard and the enterprise KPI, but labelled the difference as an
`intentional_semantic_difference` rather than recognizing the Growth
definition as stale.

### 3. Confidence calibration

Mean verification confidence was approximately `0.968`.

This is high relative to the observed factual completeness and indicates
that future work should calibrate confidence against measured answer quality.

### 4. Revision behavior

No supported evaluation question triggered the bounded revision loop.

The revision mechanism is implemented and tested deterministically, but the
current benchmark did not naturally exercise it.

### 5. Latency

Supported questions required approximately `65 seconds` on average because
the workflow performs sequential investigation and verification LLM calls.

Unsupported questions averaged approximately `3.45 seconds` because the
relevance gate exits before LLM execution.

### 6. NLI evaluator reliability

The auxiliary NLI evaluator reported only `48.57%` required-fact entailment.

Manual auditing exposed substantial false negatives, including explicit
facts being classified as neutral or contradictory.

The NLI result is therefore retained as an evaluator diagnostic and is not
used as the primary factual-accuracy metric.

In [108]:
# Building the final per-question evaluation report.

per_question_evaluation_df = (
    evaluation_run_df
    .merge(
        audited_question_fact_df,
        on="question_id",
        how="left",
    )
    .merge(
        classification_df[
            [
                "question_id",
                "expected_classification",
                "classification_correct",
            ]
        ],
        on="question_id",
        how="left",
    )
)


display(
    per_question_evaluation_df
)

,question_id,category,decision,diagnosis,confidence,top1_relevance,revisions,llm_calls,latency_seconds,source_count,error,required_facts,supported_facts,all_facts_supported,fact_coverage,expected_classification,classification_correct
0,Q001,conflict_detection,approved,metric_migration,0.95,0.993834,0,2,48.476258,5,None,4,4,True,1.000000,stale_version_conflict,True
1,Q002,version_detection,approved,metric_migration,1.00,0.987470,0,2,43.830111,4,None,3,2,False,0.666667,None,None
2,Q003,staleness_detection,approved,version_mismatch,0.95,0.998704,0,2,50.765382,5,None,3,3,True,1.000000,stale_version_conflict,True
3,Q004,semantic_conflict,approved,intentional_semantic_difference,0.95,0.992669,0,2,85.390344,5,None,3,2,False,0.666667,stale_semantic_definition,False
4,Q005,semantic_classification,approved,intentional_semantic_difference,1.00,0.932100,0,2,51.428417,5,None,4,4,True,1.000000,expected_semantic_difference,True
5,Q006,version_detection,approved,metric_migration,0.95,0.996773,0,2,148.333720,5,None,3,3,True,1.000000,None,None
6,Q007,metric_migration,approved,metric_migration,0.98,0.995834,0,2,82.660518,5,None,4,4,True,1.000000,expected_metric_migration,True
7,Q008,lineage,approved,metric_migration,0.95,0.807955,0,2,46.883494,3,None,4,1,False,0.250000,None,None
8,Q009,impact_analysis,approved,version_mismatch,0.95,0.920117,0,2,53.615511,5,None,3,2,False,0.666667,None,None
9,Q010,version_detection,approved,metric_migration,1.00,0.989857,0,2,39.888605,5,None,4,4,True,1.000000,None,None


In [109]:
# Creating the formal evaluation output directory.

from pathlib import Path


EVALUATION_OUTPUT_DIR = (
    REPO_DIR
    / "outputs"
    / "evaluation"
)


EVALUATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "✅ Evaluation output directory:",
    EVALUATION_OUTPUT_DIR
)

✅ Evaluation output directory: /content/metricguard-ai/outputs/evaluation


In [110]:
# Saving the final benchmark summary as JSON.

import json


summary_path = (
    EVALUATION_OUTPUT_DIR
    / "evaluation_summary.json"
)


with summary_path.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        final_evaluation_metrics,
        file,
        indent=2,
        ensure_ascii=False,
    )


print(
    "✅ Saved:",
    summary_path
)

✅ Saved: /content/metricguard-ai/outputs/evaluation/evaluation_summary.json


In [111]:
# Saving detailed evaluation tables as CSV files.

artifact_tables = {

    "headline_metrics.csv":
        headline_metrics_df,

    "supported_question_results.csv":
        per_question_evaluation_df,

    "fact_audit.csv":
        final_fact_audit_df,

    "missed_facts.csv":
        final_missed_facts_df,

    "classification_results.csv":
        classification_df,

    "unsupported_safety_results.csv":
        unsupported_results_df,
}


for file_name, dataframe in (
    artifact_tables.items()
):

    output_path = (
        EVALUATION_OUTPUT_DIR
        / file_name
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )

    print(
        "✅ Saved:",
        output_path
    )

✅ Saved: /content/metricguard-ai/outputs/evaluation/headline_metrics.csv
✅ Saved: /content/metricguard-ai/outputs/evaluation/supported_question_results.csv
✅ Saved: /content/metricguard-ai/outputs/evaluation/fact_audit.csv
✅ Saved: /content/metricguard-ai/outputs/evaluation/missed_facts.csv
✅ Saved: /content/metricguard-ai/outputs/evaluation/classification_results.csv
✅ Saved: /content/metricguard-ai/outputs/evaluation/unsupported_safety_results.csv


In [112]:
# Validating the saved evaluation artifacts.

expected_evaluation_artifacts = [
    "evaluation_summary.json",
    "headline_metrics.csv",
    "supported_question_results.csv",
    "fact_audit.csv",
    "missed_facts.csv",
    "classification_results.csv",
    "unsupported_safety_results.csv",
]


for file_name in (
    expected_evaluation_artifacts
):

    path = (
        EVALUATION_OUTPUT_DIR
        / file_name
    )

    assert (
        path.exists()
    )

    assert (
        path.stat().st_size
        > 0
    )

    print(
        "PASS:",
        file_name
    )


print(
    "\n🔥 FORMAL EVALUATION ARTIFACTS VALIDATED."
)

PASS: evaluation_summary.json
PASS: headline_metrics.csv
PASS: supported_question_results.csv
PASS: fact_audit.csv
PASS: missed_facts.csv
PASS: classification_results.csv
PASS: unsupported_safety_results.csv

🔥 FORMAL EVALUATION ARTIFACTS VALIDATED.


In [113]:
# Validating the complete Phase 9 benchmark.

assert (
    final_supported
    == 29
)

assert (
    final_missing
    == 6
)

assert (
    final_incorrect
    == 0
)

assert round(
    final_fact_accuracy,
    4,
) == 0.8286

assert round(
    full_question_accuracy,
    2,
) == 0.60

assert round(
    classification_accuracy,
    2,
) == 0.80

assert (
    unsupported_rejection_rate
    == 1.0
)

assert (
    false_acceptance_rate
    == 0.0
)

assert (
    zero_llm_rate
    == 1.0
)

assert (
    maximum_unsupported_score
    <
    relevance_config
    .top1_rerank_threshold
)


print(
    "=" * 80
)

print(
    "🔥 PHASE 9 FORMAL EVALUATION PASSED."
)

print(
    "=" * 80
)

print(
    "✅ 10 supported questions evaluated"
)

print(
    "✅ 35 required facts adjudicated"
)

print(
    "✅ 82.86% factual coverage"
)

print(
    "✅ 60.00% fully complete answers"
)

print(
    "✅ 80.00% classification accuracy"
)

print(
    "✅ 100% supported retrieval acceptance"
)

print(
    "✅ 100% source presence"
)

print(
    "✅ 100% unsupported rejection"
)

print(
    "✅ 0% unsupported false acceptance"
)

print(
    "✅ 100% unsupported zero-LLM execution"
)

print(
    "✅ Formal evaluation artifacts saved"
)

🔥 PHASE 9 FORMAL EVALUATION PASSED.
✅ 10 supported questions evaluated
✅ 35 required facts adjudicated
✅ 82.86% factual coverage
✅ 60.00% fully complete answers
✅ 80.00% classification accuracy
✅ 100% supported retrieval acceptance
✅ 100% source presence
✅ 100% unsupported rejection
✅ 0% unsupported false acceptance
✅ 100% unsupported zero-LLM execution
✅ Formal evaluation artifacts saved


# Summarizing the completed formal evaluation

## Phase 9 Summary — Formal Evaluation Complete

MetricGuard was evaluated against quarantined synthetic ground truth using
10 supported analytics-governance questions, 35 required facts, and 6
unsupported out-of-domain questions.

### Answer Quality

- Execution success: `100%`
- Human-audited factual coverage: `82.86%` (`29 / 35`)
- Fully complete answers: `60%` (`6 / 10`)
- Classification accuracy: `80%` (`4 / 5`)
- Explicitly incorrect audited facts: `0`
- Missing required facts: `6`

### Retrieval and Evidence

- Supported retrieval acceptance: `100%`
- Source presence: `100%`
- Calibrated relevance threshold: `0.27`

### Unsupported-Query Safety

- Unsupported questions evaluated: `6`
- Rejection rate: `100%`
- False acceptance rate: `0%`
- Zero-LLM execution rate: `100%`
- Maximum unsupported Top-1 rerank score: `0.012`

### Runtime

- Mean supported-query latency: approximately `65.13 seconds`
- Mean unsupported-query latency: approximately `3.45 seconds`
- Supported benchmark LLM calls: `20`
- Revision rate: `0%`

### Key Failure Modes

1. Full lineage answers may terminate before reaching all upstream raw sources.
2. Stale semantic definitions may be classified as intentional semantic differences.
3. Verification confidence is high relative to measured factual completeness.
4. The natural benchmark did not trigger the bounded revision loop.

### Evaluation Methodology Note

An independent NLI evaluator was tested for semantic fact scoring but
produced substantial false negatives.

Its `48.57%` entailment result is retained only as an auxiliary evaluator
diagnostic.

The primary factual benchmark is the completed ground-truth fact audit.

### Evaluation Artifacts

Formal benchmark artifacts are stored under:

`outputs/evaluation/`

### Next Phase

Production application and deployment preparation.

In [115]:
# Packaging formal evaluation artifacts for local repository sync.

import shutil

from google.colab import files


evaluation_archive = shutil.make_archive(
    "/content/metricguard_phase9_evaluation",
    "zip",
    root_dir=EVALUATION_OUTPUT_DIR,
)


print(
    "✅ Evaluation archive created:",
    evaluation_archive
)


files.download(
    evaluation_archive
)

✅ Evaluation archive created: /content/metricguard_phase9_evaluation.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>